# 01 — Raccolta dati grezzi (Blocco A)

Obiettivo di questo notebook: scaricare/organizzare i documenti grezzi per ciascuna dimensione (instabilita', carestia, migrazione, economia, cyber, contesto) e registrarli in `data/manifest.csv`.

Non fare qui l'estrazione con l'LMM: quella va nel notebook `02_extraction_lmm.ipynb`.

In [26]:
import sys
sys.path.append('..')

from src.utils.config_loader import load_countries, load_period_range

paesi = load_countries()
periodo = load_period_range()

print(f"Periodo: {periodo['inizio']} -> {periodo['fine']} ({periodo['granularita']})")
print(f"Numero paesi: {len(paesi)}")
for p in paesi:
    print(f"  {p['iso3']}: {p['nome']} (gruppo {p['gruppo']}, {p['tag']})")

Periodo: 2018-Q1 -> 2024-Q4 (trimestrale)
Numero paesi: 17
  RUS: Russia (gruppo 1, attore_cyber)
  CHN: Cina (gruppo 1, attore_cyber)
  PRK: Corea del Nord (gruppo 1, attore_cyber)
  IRN: Iran (gruppo 1, attore_cyber)
  UKR: Ucraina (gruppo 2, instabilita)
  SDN: Sudan (gruppo 2, instabilita)
  SSD: Sud Sudan (gruppo 2, instabilita)
  YEM: Yemen (gruppo 2, instabilita)
  SYR: Siria (gruppo 2, instabilita)
  ETH: Etiopia (gruppo 2, instabilita)
  VEN: Venezuela (gruppo 2, instabilita)
  USA: Stati Uniti (gruppo 3, vittima_cyber)
  ISR: Israele (gruppo 3, vittima_cyber)
  KOR: Corea del Sud (gruppo 3, vittima_cyber)
  SAU: Arabia Saudita (gruppo 3, vittima_cyber)
  ITA: Italia (gruppo 4, controllo)
  EST: Estonia (gruppo 4, controllo)


## Dimensione 1 — Instabilita'/Conflitti (ACLED)

Passi:
1. Registrarsi su ACLED Access Portal e generare la access key (salvarla in `.env`, MAI nel codice)
2. Scaricare export CSV per ciascun paese nel periodo di riferimento
3. Salvare in `data/raw/acled/<ISO3>/`
4. Aggiornare il manifest (vedi cella sotto per la funzione di logging)

In [27]:
import csv
from datetime import date
from pathlib import Path

MANIFEST_PATH = Path('../data/manifest.csv')

def log_manifest(fonte, paese_iso3, periodo, nome_file, url_originale, note=''):
    """Aggiunge una riga al manifest ogni volta che scarichi un file.
    Chiamare questa funzione subito dopo ogni download, non a fine giornata:
    e' piu' facile dimenticarsene dopo.
    """
    with open(MANIFEST_PATH, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([fonte, paese_iso3, periodo, nome_file, url_originale, date.today().isoformat(), note])

# Esempio d'uso (decommentare e adattare):
# log_manifest('acled', 'SDN', '2018-2024', 'SDN_2018-2024_acled-export.csv', 'https://acleddata.com/...', 'export CSV completo')

In [28]:
import pandas as pd
from pathlib import Path

# --- 1. File mensile per eventi ---
df_eventi = pd.read_excel('../data/raw/acled/_grezzo_mondiale/number_of_political_violence_events_by_country-month-year_as-of-26Jun2026.xlsx')

print(df_eventi['COUNTRY'].unique())  # controlla i nomi esatti prima di mappare

mappa_iso3 = {
    'Russia': 'RUS', 'China': 'CHN', 'North Korea': 'PRK', 'Iran': 'IRN',
    'Ukraine': 'UKR', 'Sudan': 'SDN', 'South Sudan': 'SSD', 'Yemen': 'YEM', 'Syria': 'SYR',
    'Ethiopia': 'ETH', 'Venezuela': 'VEN', 'United States': 'USA',
    'Israel': 'ISR', 'South Korea': 'KOR', 'Saudi Arabia': 'SAU',
    'Italy': 'ITA', 'Estonia': 'EST'
}

# Mese (nome inglese) -> trimestre
df_eventi['QUARTER'] = pd.to_datetime(df_eventi['MONTH'], format='%B').dt.quarter
df_eventi['PERIODO'] = df_eventi['YEAR'].astype(str) + '-Q' + df_eventi['QUARTER'].astype(str)

# Filtra 2018-2024
df_eventi = df_eventi[df_eventi['YEAR'].between(2018, 2024)]

for nome_acled, iso3 in mappa_iso3.items():
    sotto = df_eventi[df_eventi['COUNTRY'] == nome_acled].copy()
    if sotto.empty:
        print(f"ATTENZIONE: nessun dato trovato per {nome_acled} — controlla il nome esatto in COUNTRY")
        continue
    # Aggrega da mensile a trimestrale
    trimestrale = sotto.groupby('PERIODO', as_index=False)['EVENTS'].sum()
    out_path = Path(f'../data/raw/acled/{iso3}/{iso3}_eventi-trimestrali_2018-2024.csv')
    trimestrale.to_csv(out_path, index=False)
    log_manifest('acled', iso3, '2018-2024', out_path.name,
                  'aggregated data - political violence events by country-month-year')
    print(f'{iso3}: salvato, {len(trimestrale)} trimestri')

<StringArray>
[          'Afghanistan', 'Akrotiri and Dhekelia',               'Albania',
               'Algeria',        'American Samoa',               'Andorra',
                'Angola',              'Anguilla',            'Antarctica',
   'Antigua and Barbuda',
 ...
               'Vanuatu',          'Vatican City',             'Venezuela',
               'Vietnam',  'Virgin Islands, U.S.',     'Wallis and Futuna',
                 'Yemen',                'Zambia',              'Zimbabwe',
              'eSwatini']
Length: 250, dtype: str
RUS: salvato, 28 trimestri
CHN: salvato, 28 trimestri
PRK: salvato, 23 trimestri
IRN: salvato, 28 trimestri
UKR: salvato, 28 trimestri
SDN: salvato, 28 trimestri
SSD: salvato, 28 trimestri
YEM: salvato, 28 trimestri
SYR: salvato, 28 trimestri
ETH: salvato, 28 trimestri
VEN: salvato, 28 trimestri
USA: salvato, 20 trimestri
ISR: salvato, 28 trimestri
KOR: salvato, 21 trimestri
SAU: salvato, 26 trimestri
ITA: salvato, 20 trimestri
EST: salvato, 8 t

In [29]:
# Per ogni paese, controlla l'anno minimo presente nel dataset grezzo mondiale
for nome_acled, iso3 in mappa_iso3.items():
    sotto = df_eventi[df_eventi['COUNTRY'] == nome_acled]
    if not sotto.empty:
        print(f"{iso3}: copertura da {sotto['YEAR'].min()} a {sotto['YEAR'].max()}, {len(sotto)} righe mensili")

RUS: copertura da 2018 a 2024, 84 righe mensili
CHN: copertura da 2018 a 2024, 84 righe mensili
PRK: copertura da 2018 a 2024, 62 righe mensili
IRN: copertura da 2018 a 2024, 84 righe mensili
UKR: copertura da 2018 a 2024, 84 righe mensili
SDN: copertura da 2018 a 2024, 84 righe mensili
SSD: copertura da 2018 a 2024, 84 righe mensili
YEM: copertura da 2018 a 2024, 84 righe mensili
SYR: copertura da 2018 a 2024, 84 righe mensili
ETH: copertura da 2018 a 2024, 84 righe mensili
VEN: copertura da 2018 a 2024, 84 righe mensili
USA: copertura da 2020 a 2024, 60 righe mensili
ISR: copertura da 2018 a 2024, 84 righe mensili
KOR: copertura da 2018 a 2024, 54 righe mensili
SAU: copertura da 2018 a 2024, 64 righe mensili
ITA: copertura da 2020 a 2024, 60 righe mensili
EST: copertura da 2020 a 2021, 24 righe mensili


In [30]:
import pandas as pd
from pathlib import Path

# --- 1. Caricamento File Annuale Vittime ---
# Modificato il percorso con il nuovo nome del file
df_fatalities = pd.read_excel('../data/raw/acled/_grezzo_mondiale/number_of_reported_fatalities_by_country-year_as-of-26Jun2026.xlsx')

# TI CONSIGLIO DI DECOMMENTARE QUESTA RIGA AL PRIMO AVVIO:
# print(df_fatalities.columns)  # <-- Controlla se si chiama 'FATALITIES' o in un altro modo!
# print(df_fatalities['COUNTRY'].unique()) 

mappa_iso3 = {
    'Russia': 'RUS', 'China': 'CHN', 'North Korea': 'PRK', 'Iran': 'IRN',
    'Ukraine': 'UKR', 'Sudan': 'SDN', 'South Sudan': 'SSD', 'Yemen': 'YEM', 'Syria': 'SYR',
    'Ethiopia': 'ETH', 'Venezuela': 'VEN', 'United States': 'USA',
    'Israel': 'ISR', 'South Korea': 'KOR', 'Saudi Arabia': 'SAU',
    'Italy': 'ITA', 'Estonia': 'EST'
}

# RIMOZIONE TRIMESTRI: Ora il PERIODO coincide semplicemente con l'ANNO
df_fatalities['PERIODO'] = df_fatalities['YEAR'].astype(str)

# Filtra gli anni 2018-2024
df_fatalities = df_fatalities[df_fatalities['YEAR'].between(2018, 2024)]

for nome_acled, iso3 in mappa_iso3.items():
    sotto = df_fatalities[df_fatalities['COUNTRY'] == nome_acled].copy()
    if sotto.empty:
        print(f"ATTENZIONE: nessun dato trovato per {nome_acled} — controlla il nome esatto in COUNTRY")
        continue
        
    # MODIFICATO: Raggruppa per anno e somma le VITTIME ('FATALITIES') invece degli eventi
    annuale = sotto.groupby('PERIODO', as_index=False)['FATALITIES'].sum()
    
    # MODIFICATO: Nuovo nome del file di output per distinguerlo da quello degli eventi
    out_path = Path(f'../data/raw/acled/{iso3}/{iso3}_vittime-annuali_2018-2024.csv')
    annuale.to_csv(out_path, index=False)
    
    # MODIFICATO: Aggiornato il manifest con la descrizione corretta
    log_manifest('acled', iso3, '2018-2024', out_path.name,
                 'aggregated data - reported fatalities by country-year')
                 
    print(f'{iso3}: salvato, {len(annuale)} anni registrati')

RUS: salvato, 7 anni registrati
CHN: salvato, 7 anni registrati
PRK: salvato, 7 anni registrati
IRN: salvato, 7 anni registrati
UKR: salvato, 7 anni registrati
SDN: salvato, 7 anni registrati
SSD: salvato, 7 anni registrati
YEM: salvato, 7 anni registrati
SYR: salvato, 7 anni registrati
ETH: salvato, 7 anni registrati
VEN: salvato, 7 anni registrati
USA: salvato, 5 anni registrati
ISR: salvato, 7 anni registrati
KOR: salvato, 7 anni registrati
SAU: salvato, 7 anni registrati
ITA: salvato, 5 anni registrati
EST: salvato, 5 anni registrati


In [31]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()  # legge ACLED_EMAIL e ACLED_PASSWORD da .env (mai hardcodare le credenziali)

def get_access_token(username, password):
    resp = requests.post(
        "https://acleddata.com/oauth/token",
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        data={'username': username, 'password': password,
              'grant_type': 'password', 'client_id': 'acled'}
    )
    resp.raise_for_status()
    return resp.json()['access_token']

token = get_access_token(os.environ['ACLED_EMAIL'], os.environ['ACLED_PASSWORD'])

In [32]:
nomi_acled = {
    'Russia': 'RUS', 'China': 'CHN', 'North Korea': 'PRK', 'Iran': 'IRN',
    'Ukraine': 'UKR', 'Sudan': 'SDN', 'South Sudan': 'SSD', 'Yemen': 'YEM',
    'Syria': 'SYR', 'Ethiopia': 'ETH', 'Venezuela': 'VEN', 'United States': 'USA',
    'Israel': 'ISR', 'South Korea': 'KOR', 'Saudi Arabia': 'SAU',
    'Italy': 'ITA', 'Estonia': 'EST'
}

params = {
    '_format': 'json',
    'country': '|'.join(nomi_acled.keys()),
    'year': '2018|2024',
    'year_where': 'BETWEEN',
    'fields': 'event_date|year|country|event_type|sub_event_type|actor1|actor2|fatalities|notes|admin1',
    'limit': 0,
}

resp = requests.get("https://acleddata.com/api/acled/read",
                     headers={'Authorization': f'Bearer {token}'},
                     params=params)
dati = resp.json()
print(dati.get('count'))  # controlla quanti record totali dice il server
df = pd.DataFrame(dati['data'])
print(df.shape, df.columns.tolist())

None
(681410, 10) ['event_date', 'year', 'event_type', 'sub_event_type', 'actor1', 'actor2', 'country', 'admin1', 'notes', 'fatalities']


In [33]:
print(df['country'].value_counts())          # vedi quanti eventi per paese: sono tutti e 17 presenti?
print(df['event_date'].min(), df['event_date'].max())  # copre davvero 2018-2024 o si ferma prima?

country
Ukraine          208236
Syria            105441
United States     74722
Yemen             72267
South Korea       39237
Russia            28990
Venezuela         26434
Italy             25064
Iran              24790
Sudan             22077
Israel            15950
China             15803
Ethiopia          10231
South Sudan        9805
Saudi Arabia       1608
Estonia             389
North Korea         366
Name: count, dtype: int64
2018-01-01 2024-12-31


In [34]:
from pathlib import Path

# 1. Aggiungi una colonna trimestre, ci servirà nel Blocco A
df['PERIODO'] = pd.to_datetime(df['event_date']).dt.to_period('Q').astype(str)
# esempio di risultato: "2020Q1" -> se vuoi il formato con trattino: .str.replace('Q', '-Q')
df['PERIODO'] = df['PERIODO'].str.replace('Q', '-Q')

# 2. Dividi per paese e salva, esattamente come abbiamo già fatto altre volte
for nome_acled, iso3 in nomi_acled.items():
    sotto = df[df['country'] == nome_acled].copy()
    if sotto.empty:
        print(f"ATTENZIONE: nessun evento trovato per {nome_acled}")
        continue
    out_path = Path(f'../data/raw/acled/{iso3}/{iso3}_eventi-dettagliati_2018-2024.csv')
    sotto.to_csv(out_path, index=False)
    log_manifest('acled', iso3, '2018-2024', out_path.name,
                  'API ACLED - eventi disaggregati con notes narrative')
    print(f'{iso3}: {len(sotto)} eventi salvati')

RUS: 28990 eventi salvati
CHN: 15803 eventi salvati
PRK: 366 eventi salvati
IRN: 24790 eventi salvati
UKR: 208236 eventi salvati
SDN: 22077 eventi salvati
SSD: 9805 eventi salvati
YEM: 72267 eventi salvati
SYR: 105441 eventi salvati
ETH: 10231 eventi salvati
VEN: 26434 eventi salvati
USA: 74722 eventi salvati
ISR: 15950 eventi salvati
KOR: 39237 eventi salvati
SAU: 1608 eventi salvati
ITA: 25064 eventi salvati
EST: 389 eventi salvati


### Complemento multimodale: report PDF ACAPS (instabilita'/conflitti)

ACLED (sopra) copre il segnale quantitativo (eventi, vittime). Per la
componente multimodale della tesi aggiungiamo qui i report di analisi
PDF di [ACAPS](https://www.acaps.org/en/countries) sullo stesso tema:
crisi umanitarie, accesso, scenari di rischio, per paese.

Copertura: ACAPS non e' un prodotto onnicomprensivo come ACLED, copre
solo i paesi con una crisi umanitaria attiva. Su 17 paesi del progetto,
10 hanno una country page ACAPS con almeno un report nel 2018-2024
(CHN, IRN, UKR, SDN, SSD, YEM, SYR, ETH, VEN, ITA); gli altri 7 (RUS,
PRK, USA, ISR, KOR, SAU, EST) non hanno alcun prodotto ACAPS dedicato —
gap reale della fonte, non un bug.

L'archivio paese (`/en/countries/archives`) e' paginato via TYPO3
(`tx_acapspackage_dataproductlist[...]`); i nomi dei PDF iniziano con
la data `YYYYMMDD_`, usata per filtrare al periodo 2018-2024 e per
capire quando fermare la paginazione (pagina piu' vecchia del 2018).


In [ ]:
import re
import time
from datetime import date
from pathlib import Path

import requests

HEADERS = {"User-Agent": "ThesisResearchBot/1.0 (progetto di tesi accademico; contatto: giacomomaldarella9@gmail.com)"}
BASE = "https://www.acaps.org"

# id numerico del paese nel sistema ACAPS (trovato nel link "archives" di
# ogni country page, es. acaps.org/en/countries/sudan -> country=385).
# Solo i paesi con una country page ACAPS attiva compaiono qui: gli altri
# 7 del progetto (RUS, PRK, USA, ISR, KOR, SAU, EST) non hanno prodotti
# ACAPS dedicati.
PAESI_ACAPS = {
    'CHN': 236, 'IRN': 299, 'UKR': 423, 'SDN': 385, 'SSD': 398,
    'YEM': 437, 'SYR': 406, 'ETH': 263, 'VEN': 430, 'ITA': 303,
}

DATA_INIZIO = date(2018, 1, 1)
DATA_FINE = date(2024, 12, 31)


def elenca_pdf_paese(country_id, max_pagine=40):
    trovati = {}
    for pagina in range(1, max_pagine + 1):
        url = (f"{BASE}/en/countries/archives?"
               f"tx_acapspackage_dataproductlist%5Bcontroller%5D=DataProduct"
               f"&tx_acapspackage_dataproductlist%5BcurrentPage%5D={pagina}"
               f"&tx_acapspackage_dataproductlist%5Bfilter%5D%5Bcountry%5D={country_id}")
        r = requests.get(url, headers=HEADERS, timeout=20)
        pdf_links = sorted(set(re.findall(r'href="(/fileadmin/[^"]+\.pdf)"', r.text)))
        if not pdf_links:
            break
        date_pagina = []
        for link in pdf_links:
            m = re.search(r'/(\d{8})_', link)
            if not m:
                continue
            try:
                d = date(int(m.group(1)[:4]), int(m.group(1)[4:6]), int(m.group(1)[6:8]))
            except ValueError:
                continue
            date_pagina.append(d)
            if DATA_INIZIO <= d <= DATA_FINE:
                trovati[link] = d
        if date_pagina and max(date_pagina) < DATA_INIZIO:
            break
        time.sleep(0.3)
    return trovati


riepilogo = []
for iso3, country_id in PAESI_ACAPS.items():
    out_dir = Path(f"../data/raw/acaps/{iso3}/")
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_map = elenca_pdf_paese(country_id)
    trovati = 0
    for link, d in sorted(pdf_map.items(), key=lambda x: x[1]):
        r = requests.get(BASE + link, headers=HEADERS, timeout=30)
        if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/pdf"):
            nome = f"{iso3}_{d.isoformat()}_{link.rsplit('/', 1)[-1]}"
            (out_dir / nome).write_bytes(r.content)
            trovati += 1
        time.sleep(0.3)
    print(f"{iso3}: {trovati}/{len(pdf_map)} pdf scaricati")
    riepilogo.append((iso3, trovati))

print()
print("Totale pdf ACAPS scaricati:", sum(n for _, n in riepilogo))


## Dimensione 2 — Carestia/Siccita' (FEWS NET)



In [35]:
from dbfread import DBF
import pandas as pd

# Prendi un file .dbf a caso dalla cartella East Africa per vedere le colonne
tabella = DBF('../data/raw/fews_net/_bulk/ALL_HFIC/East Africa/EA_201802_CS.dbf', encoding='latin-1')
df_esempio = pd.DataFrame(iter(tabella))
print(df_esempio.columns.tolist())
df_esempio.head()

['CS', 'HA0']


,CS,HA0
0,1.0,0.0
1,1.0,1.0
2,2.0,0.0
3,2.0,1.0
4,3.0,0.0


In [36]:
import geopandas as gpd

gdf = gpd.read_file('../data/raw/fews_net/_bulk/ALL_HFIC/East Africa/EA_202202_CS.shp')
print(gdf.columns.tolist())
gdf.head()

['CS', 'HA0', 'geometry']


,CS,HA0,geometry
0,1,0,"MULTIPOLYGON (((35.56812 5.40842, 35.56399 5.4..."
1,1,0,"MULTIPOLYGON (((29.66249 -4.42669, 29.66287 -4..."
2,2,0,"MULTIPOLYGON (((30.05849 5.08971, 30.10733 4.9..."
3,2,0,"MULTIPOLYGON (((25.07247 10.25698, 25.07223 10..."
4,2,1,"MULTIPOLYGON (((23.38489 11.22949, 23.38153 11..."


### Pivot: report per paese in PDF invece del bulk ALL_HFIC

Il bulk ALL_HFIC (celle sopra) e' stato scartato: si fermava al 2021 e le
tabelle non identificavano chiaramente il paese. Alternativa trovata:
ogni pagina report di fews.net ha una versione `/print` che restituisce
il PDF vero (content-type application/pdf) - niente KMU presi a mano.
Pattern URL: `/{regione}/{paese}/{tipo-report}/{mese}-{anno}/print`.

Solo 7 dei 17 paesi del progetto sono coperti da FEWS NET (gruppo
"instabilita'": UKR, SDN, SSD, YEM, SYR, ETH, VEN) - gli altri 10 non hanno
pagina paese su questa fonte, coerente con quanto già verificato a mano.

Attenzione: lo slug "corto" di atterraggio (es. `/venezuela`,
`/global/ukraine`) non è detto sia il prefisso reale dei report - per
UKR e VEN i report vivono sotto un prefisso regionale diverso
(`middle-east-and-europe/ukraine`, `latin-america-and-caribbean/venezuela`),
scoperto solo cercando i link ai singoli report sulla pagina paese.

In [ ]:
import time
from pathlib import Path

import requests

HEADERS = {"User-Agent": "ThesisResearchBot/1.0 (progetto di tesi accademico; contatto: giacomomaldarella9@gmail.com)"}
BASE = "https://fews.net"

# Sostituisce il bulk ALL_HFIC (shapefile, fermo al 2021, paese non chiaro
# nelle tabelle): scarichiamo direttamente i report FEWS NET per paese in
# PDF vero (endpoint "/print" di ogni pagina report, content-type
# application/pdf). Solo i 7 paesi del progetto coperti da FEWS NET
# (gruppo "instabilita'"): gli altri 10 non hanno pagina paese su FEWS NET.
#
# Nota: lo slug "corto" del paese (es. /venezuela, /global/ukraine) funziona
# come pagina di atterraggio ma NON e' detto sia il prefisso reale dei
# report - vanno verificati sulla pagina paese (cercare i link ai singoli
# report) invece di indovinare dal nome regione.
PAESI_FEWS = {
    'UKR': 'middle-east-and-europe/ukraine',
    'SDN': 'east-africa/sudan',
    'SSD': 'east-africa/south-sudan',
    'YEM': 'middle-east-and-asia/yemen',
    'SYR': 'middle-east-and-europe/syria',
    'ETH': 'east-africa/ethiopia',
    'VEN': 'latin-america-and-caribbean/venezuela',
}

MESI = ['january', 'february', 'march', 'april', 'may', 'june', 'july',
        'august', 'september', 'october', 'november', 'december']
ANNI = range(2018, 2025)

# In un dato mese FEWS NET pubblica UNO di questi tipi (si escludono a
# vicenda) - proviamo nell'ordine, il primo che risponde con un PDF vince.
# "targeted-analysis" e' usato es. dall'Ucraina per approfondimenti ad hoc.
TIPI_REPORT = ['key-message-update', 'food-security-outlook-update',
               'food-security-outlook', 'targeted-analysis']

def scarica_report_mese(slug_paese, anno, mese, tentativi=3):
    for tipo in TIPI_REPORT:
        url = f"{BASE}/{slug_paese}/{tipo}/{mese}-{anno}/print"
        for tentativo in range(tentativi):
            try:
                r = requests.get(url, headers=HEADERS, timeout=20)
            except requests.RequestException:
                time.sleep(3)
                continue
            if r.status_code == 429:
                time.sleep(int(r.headers.get("retry-after", 10)))
                continue
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/pdf"):
                return tipo, r.content
            break  # 404 o altro: non e' questo tipo di report, prova il prossimo
    return None, None

riepilogo = []
for iso3, slug in PAESI_FEWS.items():
    out_dir = Path(f"../data/raw/fews_net/{iso3}/")
    out_dir.mkdir(parents=True, exist_ok=True)
    trovati = 0
    per_tipo = {}
    for anno in ANNI:
        for mese in MESI:
            tipo, contenuto = scarica_report_mese(slug, anno, mese)
            if tipo:
                nome_file = f"{iso3}_{anno}-{mese}_{tipo}.pdf"
                (out_dir / nome_file).write_bytes(contenuto)
                trovati += 1
                per_tipo[tipo] = per_tipo.get(tipo, 0) + 1
            time.sleep(0.4)
    totale_mesi = len(list(ANNI)) * 12
    print(f"{iso3}: {trovati}/{totale_mesi} mesi coperti — {per_tipo}")
    riepilogo.append((iso3, trovati, totale_mesi))

print()
print("Totale report scaricati:", sum(t for _, t, _ in riepilogo), "su",
      sum(tot for _, _, tot in riepilogo), "mesi-paese possibili")


## Dimensione 3 — Migrazione (UNHCR)


In [37]:
import requests

# Test con un solo paese per vedere la struttura della risposta
resp = requests.get(
    "https://api.unhcr.org/population/v1/population/",
    params={
        "coo": "SUD",          # country of origin
        "yearFrom": 2018,
        "yearTo": 2024,
        "limit": 20
    }
)
print(resp.status_code)
dati = resp.json()
print(dati.keys())
print(dati.get('items', dati)[:3])  # guarda le prime righe per capire i nomi dei campi

200
dict_keys(['page', 'short-url', 'maxPages', 'total', 'items'])
[{'year': 2018, 'coo_id': 177, 'coo_name': 'Sudan', 'coo': 'SUD', 'coo_iso': 'SDN', 'coa_id': '-', 'coa_name': '-', 'coa': '-', 'coa_iso': '-', 'refugees': 724787, 'asylum_seekers': 67433, 'returned_refugees': 1804, 'idps': 1864195, 'returned_idps': '0', 'stateless': '0', 'ooc': '0', 'oip': '-', 'hst': '0'}, {'year': 2019, 'coo_id': 177, 'coo_name': 'Sudan', 'coo': 'SUD', 'coo_iso': 'SDN', 'coa_id': '-', 'coa_name': '-', 'coa': '-', 'coa_iso': '-', 'refugees': 734780, 'asylum_seekers': 72017, 'returned_refugees': 2191, 'idps': 1885782, 'returned_idps': '0', 'stateless': '0', 'ooc': 7, 'oip': '-', 'hst': '0'}, {'year': 2020, 'coo_id': 177, 'coo_name': 'Sudan', 'coo': 'SUD', 'coo_iso': 'SDN', 'coa_id': '-', 'coa_name': '-', 'coa': '-', 'coa_iso': '-', 'refugees': 787823, 'asylum_seekers': 70049, 'returned_refugees': 30, 'idps': 2552174, 'returned_idps': '0', 'stateless': '0', 'ooc': 158, 'oip': '-', 'hst': '0'}]


In [38]:
params = {
    "coo": ",".join(['SDN','SSD','RUS','CHN','PRK','IRN','UKR','YEM','SYR',
                      'ETH','VEN','USA','ISR','KOR','SAU','ITA','EST']),
    "cf_type": "ISO",   # <-- usa direttamente i nostri ISO3, niente più SUD/SDN
    "yearFrom": 2018,
    "yearTo": 2024,
    "limit": 1000
}
resp = requests.get("https://api.unhcr.org/population/v1/population/", params=params)
dati = resp.json()
df = pd.DataFrame(dati['items'])
df[['coo_iso','coo_name']].drop_duplicates()  # verifica che siano tutti e 17

,coo_iso,coo_name
0,CHN,China
1,EST,Estonia
2,ETH,Ethiopia
3,IRN,Iran (Islamic Rep. of)
4,ISR,Israel
5,ITA,Italy
6,KOR,Rep. of Korea
7,PRK,Dem. People's Rep. of Korea
8,RUS,Russian Federation
9,SAU,Saudi Arabia


In [39]:
import requests
import pandas as pd
from pathlib import Path

lista_iso3 = ['RUS','CHN','PRK','IRN','UKR','SDN','SSD','YEM','SYR','ETH',
              'VEN','USA','ISR','KOR','SAU','ITA','EST']

# --- 1. Chiamata aggregata: totale rifugiati/richiedenti asilo/IDP usciti per anno ---
params_aggregato = {
    "coo": ",".join(lista_iso3),
    "cf_type": "ISO",
    "yearFrom": 2018,
    "yearTo": 2024,
    "limit": 1000
}
resp1 = requests.get("https://api.unhcr.org/population/v1/population/", params=params_aggregato)
df_aggregato = pd.DataFrame(resp1.json()['items'])
print("Aggregato:", df_aggregato.shape)
print(df_aggregato['coo_iso'].value_counts())  # controlla che ci siano tutti e 17, ~7 righe (anni) ciascuno

# --- 2. Chiamata disaggregata per destinazione ---
params_dettaglio = {
    "coo": ",".join(lista_iso3),
    "cf_type": "ISO",
    "coa_all": "true",
    "yearFrom": 2018,
    "yearTo": 2024,
    "limit": 50000
}
resp2 = requests.get("https://api.unhcr.org/population/v1/population/", params=params_dettaglio)
df_destinazioni = pd.DataFrame(resp2.json()['items'])
print("Destinazioni:", df_destinazioni.shape)

Aggregato: (119, 18)
coo_iso
CHN    7
EST    7
ETH    7
IRN    7
ISR    7
ITA    7
KOR    7
PRK    7
RUS    7
SAU    7
SSD    7
SDN    7
SYR    7
UKR    7
USA    7
VEN    7
YEM    7
Name: count, dtype: int64
Destinazioni: (6454, 18)


In [40]:
for iso3 in lista_iso3:

    # --- File 1: flusso totale annuale (raw, così com'è dall'API) ---
    sotto_agg = df_aggregato[df_aggregato['coo_iso'] == iso3].copy()
    if sotto_agg.empty:
        print(f"ATTENZIONE: nessun dato aggregato per {iso3}")
    else:
        out_path = Path(f'../data/raw/unhcr/{iso3}/{iso3}_flusso-uscita-annuale_2018-2024.csv')
        sotto_agg.to_csv(out_path, index=False)
        log_manifest('unhcr', iso3, '2018-2024', out_path.name,
                      'API UNHCR - population, coo aggregato su tutte le destinazioni')

    # --- File 2: dettaglio destinazioni (raw) ---
    sotto_dest = df_destinazioni[df_destinazioni['coo_iso'] == iso3].copy()
    if sotto_dest.empty:
        print(f"ATTENZIONE: nessun dato destinazioni per {iso3}")
        continue
    out_path2 = Path(f'../data/raw/unhcr/{iso3}/{iso3}_destinazioni-dettaglio_2018-2024.csv')
    sotto_dest.to_csv(out_path2, index=False)
    log_manifest('unhcr', iso3, '2018-2024', out_path2.name,
                  'API UNHCR - population, coa_all=true (dettaglio per paese destinazione)')

    # --- File 3: riassunto pronto per lo schema - top 3 destinazioni per anno ---
    sotto_dest['refugees'] = pd.to_numeric(sotto_dest['refugees'], errors='coerce').fillna(0)
    top_dest = (sotto_dest
                .sort_values('refugees', ascending=False)
                .groupby('year')
                .head(3)
                .sort_values(['year', 'refugees'], ascending=[True, False]))
    out_path3 = Path(f'../data/raw/unhcr/{iso3}/{iso3}_top-destinazioni_2018-2024.csv')
    top_dest[['year', 'coa_name', 'refugees']].to_csv(out_path3, index=False)

    print(f'{iso3}: aggregato {len(sotto_agg)} righe, destinazioni {len(sotto_dest)} righe, top3 salvato')

RUS: aggregato 7 righe, destinazioni 483 righe, top3 salvato
CHN: aggregato 7 righe, destinazioni 405 righe, top3 salvato
PRK: aggregato 7 righe, destinazioni 86 righe, top3 salvato
IRN: aggregato 7 righe, destinazioni 586 righe, top3 salvato
UKR: aggregato 7 righe, destinazioni 481 righe, top3 salvato
SDN: aggregato 7 righe, destinazioni 712 righe, top3 salvato
SSD: aggregato 7 righe, destinazioni 373 righe, top3 salvato
YEM: aggregato 7 righe, destinazioni 655 righe, top3 salvato
SYR: aggregato 7 righe, destinazioni 843 righe, top3 salvato
ETH: aggregato 7 righe, destinazioni 610 righe, top3 salvato
VEN: aggregato 7 righe, destinazioni 396 righe, top3 salvato
USA: aggregato 7 righe, destinazioni 236 righe, top3 salvato
ISR: aggregato 7 righe, destinazioni 163 righe, top3 salvato
KOR: aggregato 7 righe, destinazioni 89 righe, top3 salvato
SAU: aggregato 7 righe, destinazioni 209 righe, top3 salvato
ITA: aggregato 7 righe, destinazioni 86 righe, top3 salvato
EST: aggregato 7 righe, des

In [41]:
# Espande ogni riga annuale in 4 righe trimestrali, stesso valore ripetuto,
# con un flag esplicito che dichiara la vera risoluzione del dato.
righe_trimestrali = []
for _, riga in df_aggregato.iterrows():
    for q in range(1, 5):
        righe_trimestrali.append({
            'iso3': riga['coo_iso'],
            'periodo': f"{riga['year']}-Q{q}",
            'refugees': riga['refugees'],
            'asylum_seekers': riga['asylum_seekers'],
            'idps': riga['idps'],
            'risoluzione_dato': 'annuale'   # <-- dichiarazione esplicita, non un dato "vero" trimestrale
        })

df_trimestrale = pd.DataFrame(righe_trimestrali)

for iso3 in lista_iso3:
    sotto = df_trimestrale[df_trimestrale['iso3'] == iso3]
    out_path = Path(f'../data/processed/extracted_json/{iso3}/{iso3}_migrazione_trimestrale.csv')
    sotto.to_csv(out_path, index=False)

In [ ]:
import requests

API_KEY = "la_tua_chiave"  # da salvare in .env, non hardcoded

headers = {"Authorization": f"Bearer {API_KEY}"}  # o altro schema, da confermare dalla doc

# Test su una sola situation, es. Sudan
resp = requests.get(
    "https://data.unhcr.org/api/documents",  # endpoint indicativo, da verificare sulla doc
    headers=headers,
    params={"situation": "sudan", "doc_type": "external update", "limit": 20}
)
print(resp.status_code)
print(resp.json())

### Complemento multimodale: report PDF UNHCR via ReliefWeb (migrazione)

La microdata API di UNHCR (sopra) copre il segnale quantitativo (flussi,
destinazioni). Per la componente multimodale aggiungiamo qui i report
PDF periodici pubblicati da UNHCR, recuperati non dal sito UNHCR
(bloccato da anti-bot piu' aggressivo di quello incontrato altrove, e
l'accesso all'API richiede approvazione manuale mai arrivata) ma da
[ReliefWeb](https://reliefweb.int) — l'aggregatore OCHA di report
umanitari — filtrando per paese e per fonte "UNHCR" (facet `S2868`).
ReliefWeb non richiede registrazione per la sola navigazione HTML (a
differenza della sua stessa API REST, che dal 1 novembre 2025 richiede
un `appname` pre-approvato — stesso ostacolo incontrato con l'API
UNHCR).

Trappola scoperta durante lo sviluppo: ReliefWeb blocca silenziosamente
(404 sull'allegato, nessun errore esplicito) qualsiasi User-Agent che
contenga la parola "Bot" — richiede uno User-Agent da browser reale.

Copertura: tutti i 17 paesi hanno almeno qualche report UNHCR su
ReliefWeb (anche USA, KOR, SAU, dove UNHCR opera comunque su asilo e
resettlement), ma il volume varia moltissimo con l'intensita' della
crisi migratoria del paese (es. centinaia di report per SDN/SYR/SSD,
pochissimi per KOR/SAU). Per restare coerenti con la granularita'
trimestrale del progetto e non sbilanciare il corpus, per ogni paese
si scarica **un solo PDF per trimestre** (2018-2024, 28 trimestri
possibili), scegliendo tra i candidati del trimestre quello con data
piu' vicina alla fine del trimestre stesso.


In [ ]:
import re
import threading
import queue
import time
from datetime import date, datetime
from pathlib import Path

import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "close",
}
BASE = "https://reliefweb.int"

# codice paese primario di ReliefWeb (facet PC...) — trovato sulla
# country page reliefweb.int/country/{iso3 minuscolo}
PAESI_PC = {
    'RUS': 'PC197', 'CHN': 'PC58', 'PRK': 'PC74', 'IRN': 'PC121', 'UKR': 'PC241',
    'SDN': 'PC220', 'SSD': 'PC8657', 'YEM': 'PC255', 'SYR': 'PC226', 'ETH': 'PC87',
    'VEN': 'PC250', 'USA': 'PC245', 'ISR': 'PC125', 'KOR': 'PC194', 'SAU': 'PC207',
    'ITA': 'PC126', 'EST': 'PC86',
}
FONTE_UNHCR = "S2868"  # facet sorgente ReliefWeb per "UNHCR"

ARTICOLO_RE = re.compile(r'<article class="rw-river-article.*?</article>', re.S)
LINK_RE = re.compile(r'<h3 class="rw-river-article__title"[^>]*><a href="(https://reliefweb\.int/report/[^"]+)">(.*?)</a>')
DATA_RE = re.compile(r'rw-entity-meta__tag-label--published">.*?<time datetime="([^"]+)"', re.S)
DATA_POSTED_RE = re.compile(r'rw-entity-meta__tag-label--posted">.*?<time datetime="([^"]+)"', re.S)
PDF_ATTACH_RE = re.compile(r'href="(/attachments/[^"]+\.pdf)"')

DATA_INIZIO = date(2018, 1, 1)
DATA_FINE = date(2024, 12, 31)


# Dopo molte centinaia di richieste consecutive nella stessa sessione,
# alcune risposte di reliefweb.int arrivano a scatti - mai un gap abbastanza
# lungo da far scattare il timeout di `requests` (che copre solo il gap fra
# un chunk e l'altro), ma il tempo totale della richiesta puo' esplodere.
# Serve un deadline "duro" lato client: ogni richiesta gira in un thread
# "usa e getta" (non in un pool a dimensione fissa - un thread appeso
# per sempre in un pool ne esaurirebbe i worker disponibili dopo poche
# richieste bloccate, fermando tutto). Se il thread non risponde entro
# `scadenza` secondi la richiesta viene semplicemente abbandonata.
def get_con_scadenza(url, scadenza=12, **kwargs):
    risultato = queue.Queue(maxsize=1)

    def worker():
        try:
            risultato.put(requests.get(url, headers=HEADERS, timeout=scadenza, **kwargs))
        except requests.RequestException:
            risultato.put(None)

    t = threading.Thread(target=worker, daemon=True)
    t.start()
    try:
        return risultato.get(timeout=scadenza)
    except queue.Empty:
        return None


def trimestre(d):
    return (d.year, (d.month - 1) // 3 + 1)


def fine_trimestre(y, q):
    import calendar
    ultimo_mese = {1: 3, 2: 6, 3: 9, 4: 12}[q]
    ultimo_giorno = calendar.monthrange(y, ultimo_mese)[1]
    return date(y, ultimo_mese, ultimo_giorno)


def tutti_i_trimestri():
    return [(y, q) for y in range(2018, 2025) for q in range(1, 5)]


def elenca_articoli_paese(iso3, pc_code, max_pagine=180):
    trovati = []
    errori_consecutivi = 0
    for pagina in range(1, max_pagine + 1):
        url = f"{BASE}/updates?advanced-search=%28{pc_code}%29_%28{FONTE_UNHCR}%29&page={pagina}"
        r = get_con_scadenza(url)
        if r is None:
            errori_consecutivi += 1
            if errori_consecutivi >= 8:
                print(f"  [{iso3}] troppi errori di rete consecutivi, interrompo a pag. {pagina}")
                break
            time.sleep(2 * errori_consecutivi)
            continue
        errori_consecutivi = 0
        blocchi = ARTICOLO_RE.findall(r.text)
        if not blocchi:
            break
        date_pagina = []
        for b in blocchi:
            lm = LINK_RE.search(b)
            if not lm:
                continue
            url_report, titolo = lm.group(1), lm.group(2)
            dm = DATA_RE.search(b) or DATA_POSTED_RE.search(b)
            if not dm:
                continue
            d = datetime.fromisoformat(dm.group(1)).date()
            date_pagina.append(d)
            if DATA_INIZIO <= d <= DATA_FINE:
                trovati.append((d, url_report, titolo))
        if pagina % 20 == 0:
            ultima = max(date_pagina) if date_pagina else 'n/d'
            print(f"  [{iso3}] pag. {pagina}: {len(trovati)} candidati finora (ultima data pagina: {ultima})")
        if date_pagina and max(date_pagina) < DATA_INIZIO:
            break
        time.sleep(0.4)
    return trovati


def scarica_pdf_da_report(url_report):
    r = get_con_scadenza(url_report, scadenza=10)
    if r is None:
        return None
    m = PDF_ATTACH_RE.search(r.text)
    if not m:
        return None
    rp = get_con_scadenza(BASE + m.group(1), scadenza=15)
    if rp is None:
        return None
    if rp.status_code == 200 and rp.headers.get("content-type", "").startswith("application/pdf"):
        return rp.content
    return None


def slug(testo, lunghezza=60):
    s = re.sub(r'[^a-zA-Z0-9]+', '-', testo).strip('-').lower()
    return s[:lunghezza]


riepilogo = []
for iso3, pc_code in PAESI_PC.items():
    out_dir = Path(f"../data/raw/unhcr_reports/{iso3}/")
    out_dir.mkdir(parents=True, exist_ok=True)
    articoli = elenca_articoli_paese(iso3, pc_code)
    per_trimestre = {}
    for d, url_report, titolo in articoli:
        per_trimestre.setdefault(trimestre(d), []).append((d, url_report, titolo))

    scaricati = 0
    for (y, q) in tutti_i_trimestri():
        candidati = per_trimestre.get((y, q), [])
        if not candidati:
            continue
        target = fine_trimestre(y, q)
        candidati.sort(key=lambda x: abs((x[0] - target).days))
        for d, url_report, titolo in candidati[:3]:
            contenuto = scarica_pdf_da_report(url_report)
            time.sleep(0.2)
            if contenuto:
                nome = f"{iso3}_{y}Q{q}_{d.isoformat()}_{slug(titolo)}.pdf"
                (out_dir / nome).write_bytes(contenuto)
                scaricati += 1
                break
    print(f"{iso3}: {scaricati}/28 trimestri coperti (candidati totali: {len(articoli)})")
    riepilogo.append((iso3, scaricati))

print()
print("Totale pdf UNHCR/ReliefWeb scaricati:", sum(n for _, n in riepilogo))


## Dimensione 4 — Economia/Poverta' (World Bank)

*Da fare dopo la Dimensione 3.*

In [42]:
import requests
import pandas as pd

lista_iso3 = ['RUS','CHN','PRK','IRN','UKR','SDN','SSD','YEM','SYR','ETH',
              'VEN','USA','ISR','KOR','SAU','ITA','EST']

url = f"https://api.worldbank.org/v2/country/{';'.join(lista_iso3)}/indicator/SI.POV.DDAY"
params = {
    "date": "2018:2024",
    "format": "json",
    "per_page": 1000
}
resp = requests.get(url, params=params)
risposta = resp.json()

print(risposta[0])  # metadati: controlla 'total' e 'pages', per capire se serve paginare
df = pd.DataFrame(risposta[1])
print(df.shape)
print(df.columns.tolist())
df[['countryiso3code','country','date','value']].head(10)

{'page': 1, 'pages': 1, 'per_page': 1000, 'total': 119, 'sourceid': '2', 'lastupdated': '2026-07-01'}
(119, 8)
['indicator', 'country', 'countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal']


,countryiso3code,country,date,value
0,CHN,"{'id': 'CN', 'value': 'China'}",2024,NaN
1,CHN,"{'id': 'CN', 'value': 'China'}",2023,NaN
2,CHN,"{'id': 'CN', 'value': 'China'}",2022,0.0
3,CHN,"{'id': 'CN', 'value': 'China'}",2021,0.0
4,CHN,"{'id': 'CN', 'value': 'China'}",2020,0.0
5,CHN,"{'id': 'CN', 'value': 'China'}",2019,0.0
6,CHN,"{'id': 'CN', 'value': 'China'}",2018,0.7
7,EST,"{'id': 'EE', 'value': 'Estonia'}",2024,NaN
8,EST,"{'id': 'EE', 'value': 'Estonia'}",2023,0.3
9,EST,"{'id': 'EE', 'value': 'Estonia'}",2022,0.5


In [25]:
from pathlib import Path

# Pulizia: estrai il nome paese dal dizionario annidato
df['country_nome'] = df['country'].apply(lambda x: x['value'])
df['date'] = df['date'].astype(int)

for iso3 in lista_iso3:
    sotto = df[df['countryiso3code'] == iso3].copy()
    if sotto.empty:
        print(f"ATTENZIONE: nessun dato per {iso3}")
        continue

    sotto = sotto.sort_values('date')[['date', 'value', 'obs_status']]
    sotto = sotto.rename(columns={'date': 'anno', 'value': 'tasso_poverta_pct'})

    out_path = Path(f'../data/raw/worldbank/{iso3}/{iso3}_poverta-annuale_2018-2024.csv')
    sotto.to_csv(out_path, index=False)
    log_manifest('worldbank', iso3, '2018-2024', out_path.name,
                  'API World Bank - SI.POV.DDAY (poverty headcount ratio)')

    n_validi = sotto['tasso_poverta_pct'].notna().sum()
    print(f'{iso3}: {len(sotto)} anni salvati, {n_validi} con valore (resto NaN)')

RUS: 7 anni salvati, 6 con valore (resto NaN)
CHN: 7 anni salvati, 5 con valore (resto NaN)
PRK: 7 anni salvati, 0 con valore (resto NaN)
IRN: 7 anni salvati, 6 con valore (resto NaN)
UKR: 7 anni salvati, 3 con valore (resto NaN)
SDN: 7 anni salvati, 0 con valore (resto NaN)
SSD: 7 anni salvati, 0 con valore (resto NaN)
YEM: 7 anni salvati, 0 con valore (resto NaN)
SYR: 7 anni salvati, 1 con valore (resto NaN)
ETH: 7 anni salvati, 1 con valore (resto NaN)
VEN: 7 anni salvati, 0 con valore (resto NaN)
USA: 7 anni salvati, 7 con valore (resto NaN)
ISR: 7 anni salvati, 5 con valore (resto NaN)
KOR: 7 anni salvati, 4 con valore (resto NaN)
SAU: 7 anni salvati, 0 con valore (resto NaN)
ITA: 7 anni salvati, 6 con valore (resto NaN)
EST: 7 anni salvati, 6 con valore (resto NaN)


In [1]:
import requests
from pathlib import Path

base = "https://thedocs.worldbank.org/en/doc/77351105a334213c64122e44c2efe523-0500072021/related/"

regioni = ['ssa', 'mena', 'eca', 'lac', 'eap']  # verifichiamo eap/eca/mena/lac dal primo test

edizioni = [f'{periodo}{str(anno)[2:]}'
            for anno in range(2020, 2025)
            for periodo in ['sm', 'am']]

out_dir = Path('../data/raw/worldbank/_mpo_reports/')
out_dir.mkdir(parents=True, exist_ok=True)

for regione in regioni:
    for ed in edizioni:
        nome_file = f'mpo-{ed}-{regione}.pdf'
        r = requests.get(base + nome_file)
        if r.status_code == 200:
            (out_dir / nome_file).write_bytes(r.content)
            print(f'OK: {nome_file} ({len(r.content)//1024} KB)')
        else:
            print(f'MANCA ({r.status_code}): {nome_file}')

OK: mpo-sm20-ssa.pdf (13912 KB)
OK: mpo-am20-ssa.pdf (14102 KB)
OK: mpo-sm21-ssa.pdf (14744 KB)
OK: mpo-am21-ssa.pdf (14325 KB)
OK: mpo-sm22-ssa.pdf (7199 KB)
OK: mpo-am22-ssa.pdf (7191 KB)
OK: mpo-sm23-ssa.pdf (10181 KB)
OK: mpo-am23-ssa.pdf (6974 KB)
OK: mpo-sm24-ssa.pdf (6712 KB)
OK: mpo-am24-ssa.pdf (6483 KB)
OK: mpo-sm20-mena.pdf (6605 KB)
OK: mpo-am20-mena.pdf (6265 KB)
OK: mpo-sm21-mena.pdf (6239 KB)
OK: mpo-am21-mena.pdf (6262 KB)
OK: mpo-sm22-mena.pdf (3469 KB)
OK: mpo-am22-mena.pdf (3466 KB)
OK: mpo-sm23-mena.pdf (4126 KB)
OK: mpo-am23-mena.pdf (3539 KB)
OK: mpo-sm24-mena.pdf (3355 KB)
OK: mpo-am24-mena.pdf (3953 KB)
OK: mpo-sm20-eca.pdf (7213 KB)
OK: mpo-am20-eca.pdf (7000 KB)
OK: mpo-sm21-eca.pdf (7192 KB)
OK: mpo-am21-eca.pdf (7724 KB)
OK: mpo-sm22-eca.pdf (3607 KB)
OK: mpo-am22-eca.pdf (3656 KB)
OK: mpo-sm23-eca.pdf (4705 KB)
OK: mpo-am23-eca.pdf (3805 KB)
OK: mpo-sm24-eca.pdf (3656 KB)
OK: mpo-am24-eca.pdf (3462 KB)
OK: mpo-sm20-lac.pdf (8645 KB)
OK: mpo-am20-lac.pdf (90

In [2]:
import requests
from pathlib import Path

base = "https://thedocs.worldbank.org/en/doc/77351105a334213c64122e44c2efe523-0500072021/related/"

regioni = ['ssa', 'mena', 'eca', 'lac', 'eap']  # verifichiamo eap/eca/mena/lac dal primo test

edizioni = [f'{periodo}{str(anno)[2:]}'
            for anno in range(2018, 2020)
            for periodo in ['sm', 'am']]

out_dir = Path('../data/raw/worldbank/_mpo_reports/')
out_dir.mkdir(parents=True, exist_ok=True)

for regione in regioni:
    for ed in edizioni:
        nome_file = f'mpo-{ed}-{regione}.pdf'
        r = requests.get(base + nome_file)
        if r.status_code == 200:
            (out_dir / nome_file).write_bytes(r.content)
            print(f'OK: {nome_file} ({len(r.content)//1024} KB)')
        else:
            print(f'MANCA ({r.status_code}): {nome_file}')

MANCA (404): mpo-sm18-ssa.pdf
MANCA (404): mpo-am18-ssa.pdf
MANCA (404): mpo-sm19-ssa.pdf
MANCA (404): mpo-am19-ssa.pdf
MANCA (404): mpo-sm18-mena.pdf
MANCA (404): mpo-am18-mena.pdf
MANCA (404): mpo-sm19-mena.pdf
MANCA (404): mpo-am19-mena.pdf
MANCA (404): mpo-sm18-eca.pdf
MANCA (404): mpo-am18-eca.pdf
MANCA (404): mpo-sm19-eca.pdf
MANCA (404): mpo-am19-eca.pdf
MANCA (404): mpo-sm18-lac.pdf
MANCA (404): mpo-am18-lac.pdf
MANCA (404): mpo-sm19-lac.pdf
MANCA (404): mpo-am19-lac.pdf
MANCA (404): mpo-sm18-eap.pdf
MANCA (404): mpo-am18-eap.pdf
MANCA (404): mpo-sm19-eap.pdf
MANCA (404): mpo-am19-eap.pdf


In [4]:
import pdfplumber
from pathlib import Path

test_pdf = Path('../data/raw/worldbank/_mpo_reports/mpo-sm24-ssa.pdf')

with pdfplumber.open(test_pdf) as pdf:
    print(f"Totale pagine: {len(pdf.pages)}")
    # Stampiamo le prime 2 righe di ogni pagina, per vedere dove iniziano
    # le sezioni-paese e come è scritto il nome (maiuscolo? con altro testo affianco?)
    for i, page in enumerate(pdf.pages):
        testo = page.extract_text() or ''
        righe = testo.split('\n')[:4]
        print(f"--- Pagina {i} ---")
        print(righe)

Totale pagine: 100
--- Pagina 0 ---
['Sub-Saharan Africa', 'Macro Poverty Outlook', 'Country-by-country Analysis and Projections for the Developing World', 'MP']
--- Pagina 1 ---
['© 2024 International Bank for Reconstruction and Development / The World Bank', '1818 H Street NW, Washington DC 20433', 'Telephone: 202-473-1000', 'Internet: www.worldbank.org']
--- Pagina 2 ---
['Sub-Saharan', 'Africa', "Angola Côte d'Ivoire Liberia Senegal", 'Benin Equatorial Guinea Madagascar Seychelles']
--- Pagina 3 ---
['Angola’s poverty rates stand above what', 'would be expected for a country with its', 'ANGOLA Key conditions and', 'GDPlevel:asof2018,athirdlivedonless']
--- Pagina 4 ---
['40percent depreciation in May-June 2023 Food inflation, combined with a weak-', 'and widened the gap between the official ening labor market and a decline in per', 'Outlook', 'and the parallel exchange rates. The slide capita growth, suggests that poverty may']
--- Pagina 5 ---
['in 2015-22) led to rising debt leve

In [5]:
import re
import pdfplumber
from pathlib import Path

PATTERN = re.compile(r"([A-Z][A-Z',\.\s]{2,40}?)\s+Key conditions and")

# Un file di prova per regione
test_files = ['mpo-sm24-ssa.pdf', 'mpo-sm24-mena.pdf', 'mpo-sm24-eca.pdf',
              'mpo-sm24-lac.pdf', 'mpo-sm24-eap.pdf']

for nome_file in test_files:
    path = Path(f'../data/raw/worldbank/_mpo_reports/{nome_file}')
    print(f"\n=== {nome_file} ===")
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages):
            testo = page.extract_text() or ''
            m = PATTERN.search(testo)
            if m:
                print(f"  pagina {i}: {m.group(1).strip()}")


=== mpo-sm24-ssa.pdf ===
  pagina 3: ANGOLA
  pagina 5: BENIN
  pagina 7: BOTSWANA
  pagina 9: BURKINA FASO
  pagina 11: BURUNDI
  pagina 13: CABO VERDE
  pagina 15: CAMEROON
  pagina 17: CENTRAL
  pagina 19: CHAD
  pagina 21: COMOROS
  pagina 23: DEMOCRATIC
  pagina 25: REPUBLIC OF
  pagina 27: TE D'IVOIRE
  pagina 29: EQUATORIAL
  pagina 31: ERITREA
  pagina 33: ESWATINI
  pagina 35: ETHIOPIA
  pagina 37: GABON
  pagina 39: THE GAMBIA
  pagina 41: GHANA
  pagina 43: GUINEA
  pagina 45: BISSAU
  pagina 47: KENYA
  pagina 49: LESOTHO
  pagina 51: LIBERIA
  pagina 53: MADAGASCAR
  pagina 55: MALAWI
  pagina 57: MALI
  pagina 59: MAURITANIA
  pagina 61: MAURITIUS
  pagina 63: MOZAMBIQUE
  pagina 65: NAMIBIA
  pagina 67: NIGER
  pagina 69: NIGERIA
  pagina 71: RWANDA
  pagina 73: AND
  pagina 75: SENEGAL
  pagina 77: SEYCHELLES
  pagina 79: SIERRA LEONE
  pagina 81: SOMALIA
  pagina 83: SOUTH AFRICA
  pagina 85: SOUTH SUDAN
  pagina 87: SUDAN
  pagina 89: TANZANIA
  pagina 91: TOGO
  pag

In [6]:
import pdfplumber
from pathlib import Path

with pdfplumber.open('../data/raw/worldbank/_mpo_reports/mpo-sm24-mena.pdf') as pdf:
    print(pdf.pages[39].extract_text()[:300])

impactednationalgrowthin2023andex-
acerbated IRG’s fiscal and monetary chal-
REPUBLIC OF Key conditions and
lenges. Since October 2023, the escalation
challenges of the conflict in the Middle East and in
YEMEN the Red Sea, intensified by direct Houthi
involvement, further compromises the al-
Yemen's


In [8]:
import re
import pdfplumber
from pathlib import Path
from pypdf import PdfReader, PdfWriter

PATTERN = re.compile(r"([A-Z][A-Z',\.\s]{2,40}?)\s+Key conditions and")

mappa_paesi_regione = {
    'ssa': {
        'ETHIOPIA': 'ETH',
        'SOUTH SUDAN': 'SSD',
        'SUDAN': 'SDN',
    },
    'mena': {
        'IRAN, ISLAMIC': 'IRN',
        'SYRIAN ARAB': 'SYR',
        'SAUDI ARABIA': 'SAU',
        'REPUBLIC OF YEMEN': 'YEM',  
    },
    'eca': {
        'RUSSIAN': 'RUS',
        'UKRAINE': 'UKR',
    },
    'lac': {},   # Venezuela non coperto da MPO, intenzionalmente vuoto
    'eap': {
        'CHINA': 'CHN',
    },
}

reports_dir = Path('../data/raw/worldbank/_mpo_reports/')
riepilogo = []

for pdf_path in sorted(reports_dir.glob('mpo-*.pdf')):
    m = re.match(r'mpo-(sm|am)(\d{2})-(\w+)\.pdf', pdf_path.name)
    if not m:
        continue
    tipo, anno, regione = m.groups()
    paesi_regione = mappa_paesi_regione.get(regione, {})
    if not paesi_regione:
        continue

    with pdfplumber.open(pdf_path) as pdf:
        pagine_inizio = {}
        for i, page in enumerate(pdf.pages):
            testo = page.extract_text() or ''
            match_paese = PATTERN.search(testo)
            if match_paese and match_paese.group(1).strip() in paesi_regione:
                iso3 = paesi_regione[match_paese.group(1).strip()]
                pagine_inizio[iso3] = i

    reader = PdfReader(pdf_path)
    for iso3, pagina_start in pagine_inizio.items():
        writer = PdfWriter()
        for p in [pagina_start, pagina_start + 1]:
            if p < len(reader.pages):
                writer.add_page(reader.pages[p])

        out_dir = Path(f'../data/raw/worldbank/{iso3}/')
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{iso3}_mpo_{tipo}{anno}.pdf'
        with open(out_path, 'wb') as f:
            writer.write(f)
        riepilogo.append((iso3, f'{tipo}{anno}'))

# Controllo finale: quante edizioni per paese (atteso: fino a 10)
import pandas as pd
df_riepilogo = pd.DataFrame(riepilogo, columns=['iso3', 'edizione'])
print(df_riepilogo.groupby('iso3').size())

iso3
CHN    8
ETH    9
IRN    8
RUS    7
SAU    7
SDN    8
SSD    9
SYR    6
UKR    8
dtype: int64


In [9]:
import re
import pdfplumber
from pathlib import Path
from pypdf import PdfReader, PdfWriter
import pandas as pd

# Ordine intenzionale: SOUTH SUDAN prima di SUDAN, perché "SUDAN" è una
# sottostringa di "SOUTH SUDAN" e altrimenti si confonderebbero
parole_chiave = [
    ('ETH', r'\bETHIOPIA\b'),
    ('SSD', r'\bSOUTH SUDAN\b'),
    ('SDN', r'\bSUDAN\b'),
    ('IRN', r'\bIRAN\b'),
    ('SYR', r'\bSYRIA'),
    ('SAU', r'\bSAUDI ARABIA\b'),
    ('YEM', r'\bYEMEN\b'),
    ('RUS', r'\bRUSSIA'),
    ('UKR', r'\bUKRAINE\b'),
    ('CHN', r'\bCHINA\b'),
]

def trova_iso3(testo_pagina):
    testo_upper = testo_pagina.upper()
    conteggi = {}
    for iso3, pattern in parole_chiave:
        n = len(re.findall(pattern, testo_upper))
        if n > 0:
            conteggi[iso3] = conteggi.get(iso3, 0) + n
    if not conteggi:
        return None
    return max(conteggi, key=conteggi.get)  # il piu' citato = soggetto della pagina

reports_dir = Path('../data/raw/worldbank/_mpo_reports/')
riepilogo = []

for pdf_path in sorted(reports_dir.glob('mpo-*.pdf')):
    m = re.match(r'mpo-(sm|am)(\d{2})-(\w+)\.pdf', pdf_path.name)
    if not m:
        continue
    tipo, anno, regione = m.groups()

    with pdfplumber.open(pdf_path) as pdf:
        reader = PdfReader(pdf_path)
        for i, page in enumerate(pdf.pages):
            testo = page.extract_text() or ''
            if 'Key conditions and' not in testo:
                continue

            iso3 = trova_iso3(testo)
            if iso3 is None:
                continue

            writer = PdfWriter()
            for p in [i, i + 1]:
                if p < len(reader.pages):
                    writer.add_page(reader.pages[p])

            out_dir = Path(f'../data/raw/worldbank/{iso3}/')
            out_dir.mkdir(parents=True, exist_ok=True)
            out_path = out_dir / f'{iso3}_mpo_{tipo}{anno}.pdf'
            with open(out_path, 'wb') as f:
                writer.write(f)
            riepilogo.append((iso3, f'{tipo}{anno}'))

df_riepilogo = pd.DataFrame(riepilogo, columns=['iso3', 'edizione'])
print(df_riepilogo.groupby('iso3').size())

tutte_edizioni = [f'{t}{a}' for a in ['20','21','22','23','24'] for t in ['sm','am']]
for iso3, _ in parole_chiave:
    trovate = set(df_riepilogo[df_riepilogo.iso3==iso3]['edizione'])
    mancanti = set(tutte_edizioni) - trovate
    if mancanti:
        print(f"{iso3}: mancano ancora {sorted(mancanti)}")

iso3
CHN     43
ETH     25
IRN      9
RUS    130
SAU     12
SDN     14
SSD      6
SYR     10
UKR    133
YEM      9
dtype: int64
ETH: mancano ancora ['sm20']
SSD: mancano ancora ['am23', 'am24', 'sm20', 'sm24']
SDN: mancano ancora ['sm20', 'sm22']
IRN: mancano ancora ['sm20']
SYR: mancano ancora ['am20', 'am21', 'sm20', 'sm21']
SAU: mancano ancora ['am20', 'sm20']
YEM: mancano ancora ['am20', 'sm20']
RUS: mancano ancora ['am20', 'sm20']
UKR: mancano ancora ['am20', 'sm20']
CHN: mancano ancora ['am20', 'sm20']


In [ ]:
import re
import pdfplumber
from pathlib import Path
from pypdf import PdfReader, PdfWriter
import pandas as pd

# Parole chiave per l'estrazione via testo, usata solo come fallback quando
# il PDF non ha bookmark (succede nelle edizioni 2022-2024). Ordine dal piu'
# specifico al meno specifico: SOUTH SUDAN prima di SUDAN, altrimenti
# "SUDAN" matcherebbe anche dentro "SOUTH SUDAN".
parole_chiave = [
    ('SSD', r'\bSOUTH SUDAN\b'),
    ('SDN', r'\bSUDAN\b'),
    ('ETH', r'\bETHIOPIA\b'),
    ('SAU', r'\bSAUDI ARABIA\b'),
    ('SYR', r'\bSYRIA'),
    ('IRN', r'\bIRAN\b'),
    ('YEM', r'\bYEMEN\b'),
    ('RUS', r'\bRUSSIA'),
    ('UKR', r'\bUKRAINE\b'),
    ('CHN', r'\bCHINA\b'),
]

paesi_per_regione = {
    'ssa':  {'ETH', 'SSD', 'SDN'},
    'mena': {'IRN', 'SAU', 'SYR', 'YEM'},
    'eca':  {'RUS', 'UKR'},
    'eap':  {'CHN'},
    'lac':  set(),  # Venezuela non coperto da MPO, intenzionalmente vuoto
}

def mappa_da_bookmark(reader):
    """{ISO3: pagina} leggendo i bookmark nativi del PDF (edizioni 2020-2021).
    Ogni bookmark-paese e' del tipo 'mpo-sm20-china-chn_kcm final2': l'ISO3
    e' sempre l'ultimo blocco di 3 lettere dopo un trattino, quindi non
    dipende dalla posizione ordinale del paese nel report."""
    mappa = {}
    def walk(outline):
        for item in outline:
            if isinstance(item, list):
                walk(item)
                continue
            m = re.search(r'-([a-z]{3})(?:[_\s].*)?$', item.title.strip().lower())
            if not m:
                continue
            iso3 = m.group(1).upper()
            pagina = reader.get_destination_page_number(item)
            if pagina is not None and iso3 not in mappa:
                mappa[iso3] = pagina
    try:
        walk(reader.outline)
    except Exception:
        pass
    return mappa

def mappa_da_testo(pdf):
    """{ISO3: pagina} cercando il nome del paese nell'intestazione 'Key
    conditions and' (edizioni 2022-2024, senza bookmark). La ricerca e'
    ristretta a una finestra di testo intorno al marcatore, non a tutta la
    pagina: altrimenti parole come RUSSIA/UKRAINE citate nel report di un
    altro paese (per via della guerra) vengono scambiate per l'inizio
    della loro sezione."""
    mappa = {}
    for i, page in enumerate(pdf.pages):
        testo = page.extract_text() or ''
        idx = testo.find('Key conditions and')
        if idx == -1:
            continue
        finestra = testo[max(0, idx - 150):idx + 150].upper()
        for iso3, pattern in parole_chiave:
            if iso3 not in mappa and re.search(pattern, finestra):
                mappa[iso3] = i
                break
    return mappa

reports_dir = Path('../data/raw/worldbank/_mpo_reports/')
riepilogo = []
anomalie = []

for pdf_path in sorted(reports_dir.glob('mpo-*.pdf')):
    m = re.match(r'mpo-(sm|am)(\d{2})-(\w+)\.pdf', pdf_path.name)
    if not m:
        continue
    tipo, anno, regione = m.groups()
    paesi_target = paesi_per_regione.get(regione, set())
    if not paesi_target:
        continue

    reader = PdfReader(pdf_path)
    mappa = mappa_da_bookmark(reader)

    mancanti = paesi_target - mappa.keys()
    if mancanti:
        with pdfplumber.open(pdf_path) as pdf:
            mappa.update(mappa_da_testo(pdf))

    for iso3 in paesi_target:
        pagina_start = mappa.get(iso3)
        if pagina_start is None:
            anomalie.append((pdf_path.name, iso3, "paese assente da questa edizione (nessun bookmark ne' testo)"))
            continue

        writer = PdfWriter()
        for p in [pagina_start, pagina_start + 1]:
            if p < len(reader.pages):
                writer.add_page(reader.pages[p])

        out_dir = Path(f'../data/raw/worldbank/{iso3}/')
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{iso3}_mpo_{tipo}{anno}.pdf'
        with open(out_path, 'wb') as f:
            writer.write(f)
        riepilogo.append((iso3, f'{tipo}{anno}'))

df_riepilogo = pd.DataFrame(riepilogo, columns=['iso3', 'edizione'])
print(df_riepilogo.groupby('iso3').size())

print("\n--- Anomalie residue (paese verosimilmente assente dall'edizione) ---")
for a in anomalie:
    print(a)


## Dimensione 5 — Cyber (CISA / ENISA)

*Da fare dopo la Dimensione 4.*

In [15]:
import re
import requests
import pandas as pd

# Il vecchio endpoint "interactive/cyber-operations/export-incidents?_format=csv"
# non esiste piu': CFR ha rifatto il sito (era il vecchio tool Drupal, ora e'
# WordPress su www.cfr.org/cyber-operations/). Il dato pero' resta disponibile
# via REST API WordPress, solo non piu' come CSV pronto: i post sono gli
# incidenti, e campi come sponsor/vittima/categoria sono tassonomie separate
# da risolvere con il loro id (per questo servono le chiamate a parte sotto).
BASE = "https://www.cfr.org/cyber-operations/wp-json/wp/v2"

def scarica_tassonomia(nome):
    """{id: nome} per una tassonomia (es. state_sponsor), paginando se serve."""
    termini = {}
    pagina = 1
    while True:
        r = requests.get(f"{BASE}/{nome}", params={"per_page": 100, "page": pagina})
        if r.status_code != 200:
            break
        dati = r.json()
        if not dati:
            break
        for t in dati:
            termini[t["id"]] = t["name"]
        if pagina >= int(r.headers.get("X-WP-TotalPages", 1)):
            break
        pagina += 1
    return termini

tassonomie = {
    nome: scarica_tassonomia(nome)
    for nome in ["cyber_operation", "state_sponsor", "victim_category", "victim_government_response", "victim"]
}
print({k: len(v) for k, v in tassonomie.items()})

def pulisci_html(testo):
    return re.sub('<[^<]+?>', '', testo or '').strip()

incidenti = []
pagina = 1
while True:
    r = requests.get(f"{BASE}/posts", params={"per_page": 100, "page": pagina})
    if r.status_code != 200:
        break
    dati = r.json()
    if not dati:
        break
    for p in dati:
        incidenti.append({
            "id": p["id"],
            "data_pubblicazione": p["date"][:10],
            "titolo": p["title"]["rendered"],
            "categoria": ", ".join(tassonomie["cyber_operation"].get(i, "?") for i in p.get("cyber_operation", [])),
            "sponsor_stato": ", ".join(tassonomie["state_sponsor"].get(i, "?") for i in p.get("state_sponsor", [])),
            "categoria_vittima": ", ".join(tassonomie["victim_category"].get(i, "?") for i in p.get("victim_category", [])),
            "paesi_vittima": ", ".join(tassonomie["victim"].get(i, "?") for i in p.get("victim", [])),
            "risposta_governativa": ", ".join(tassonomie["victim_government_response"].get(i, "?") for i in p.get("victim_government_response", [])),
            "descrizione": pulisci_html(p["excerpt"]["rendered"]),
            "link": p["link"],
        })
    totpag = int(r.headers.get("X-WP-TotalPages", 1))
    if pagina >= totpag:
        break
    pagina += 1

df_cfr = pd.DataFrame(incidenti)
print(df_cfr.shape)
print(df_cfr.columns.tolist())
df_cfr.head(10)


{'cyber_operation': 7, 'state_sponsor': 51, 'victim_category': 4, 'victim_government_response': 2, 'victim': 128}
(865, 10)
['id', 'data_pubblicazione', 'titolo', 'categoria', 'sponsor_stato', 'categoria_vittima', 'paesi_vittima', 'risposta_governativa', 'descrizione', 'link']


,id,data_pubblicazione,titolo,categoria,sponsor_stato,categoria_vittima,paesi_vittima,risposta_governativa,descrizione,link
0,11049,2025-10-08,Allanite,Espionage,Russian Federation,Private sector,"United Kingdom, United States",Unknown,This threat actor targets business and industr...,https://www.cfr.org/cyber-operations/allanite
1,11044,2025-10-08,Trisis,Sabotage,Russian Federation,Private sector,Saudi Arabia,Unknown,This threat actor targets the Triconex safety ...,https://www.cfr.org/cyber-operations/trisis
2,11018,2025-10-08,Leafminer,,Iran (Islamic Republic of),"Government, Private sector","Afghanistan, Azerbaijan, Bahrain, Egypt, Iran ...",Unknown,This threat actor targets government organizat...,https://www.cfr.org/cyber-operations/leafminer
3,11008,2025-10-08,"Compromises of government embassies, telecommu...",Espionage,Iran (Islamic Republic of),"Government, Private sector","Pakistan, Russian Federation, Saudi Arabia, Tu...",Unknown,A group has attacked 131 victims in thirty org...,https://www.cfr.org/cyber-operations/compromis...
4,10965,2025-10-08,Kimusky,,Korea (Democratic People's Republic of),"Government, Private sector","France, Slovakia, United Kingdom, United States",,This threat actor targeted foreign ministries ...,https://www.cfr.org/cyber-operations/kimusky
5,10921,2025-10-08,Targeting of sporting and anti-doping organiza...,Espionage,Russian Federation,"Civil society, Government",,,Just prior to news reports suggesting that the...,https://www.cfr.org/cyber-operations/targeting...
6,10907,2025-10-08,APT-C-23,,"Palestine, State of",,Israel,,Previously targeted Israeli soldiers&nbsp;by p...,https://www.cfr.org/cyber-operations/apt-c-23
7,10760,2025-10-08,APT 28,,Russian Federation,"Government, Military, Private sector","Afghanistan, Armenia, Belgium, Canada, China, ...",Yes,This threat actor is linked to espionage campa...,https://www.cfr.org/cyber-operations/apt-28
8,10698,2025-10-08,Targeting of U.S. diplomats in Uganda,Espionage,Uganda,"Civil society, Government",Uganda,Unknown,Uganda was accused of using the spyware Pegasu...,https://www.cfr.org/cyber-operations/targeting...
9,10673,2025-10-08,Targeting of American defense industry,Espionage,Korea (Democratic People's Republic of),"Military, Private sector",United States,Unknown,The North Korean threat actor Lazarus Group le...,https://www.cfr.org/cyber-operations/targeting...


### Nota sulla data degli incidenti CFR

`data_pubblicazione` (dalla nuova API) e' la data CMS della voce, non
dell'evento - vedi verifica nella cella sotto. La cella recupera le date
reali dal vecchio export CSV di CFR (Wayback Machine, ultima cattura
ottobre 2019) e le riabbina per titolo, tenendo la stima da pubblicazione
solo dove non c'e' corrispondenza (soprattutto voci 2020+, fuori dalla
finestra dell'archivio). Il risultato e' tracciato in `fonte_data_evento`.

In [ ]:
import re
import pandas as pd

# Il campo "data_pubblicazione" di df_cfr NON e' la data dell'evento: e' la
# data di creazione/modifica della voce nel CMS del nuovo sito. Verifica: su
# 865 post, l'80% non cita nemmeno un anno nel testo per un controllo
# incrociato, e tra quelli che lo citano il 30% diverge di 3+ anni dalla
# "data_pubblicazione" (es. "SideWinder" creato nel 2023 ma descrive un
# attacco del 2012). Il vecchio sito Drupal aveva invece un campo Date reale
# per ogni incidente, andato perso nella migrazione a WordPress - ma resta
# recuperabile dalla Wayback Machine: e' l'ultima cattura riuscita
# dell'export CSV originale (22 ottobre 2019, "id_" nell'URL = versione
# grezza senza la toolbar di archive.org iniettata nella pagina).
url_archivio = (
    "https://web.archive.org/web/20191022065745id_/"
    "https://www.cfr.org/interactive/cyber-operations/export-incidents"
)
df_storico = pd.read_csv(url_archivio)

def norm_titolo(s):
    s = str(s).strip().lower().replace('’', "'").replace('‘', "'")
    s = re.sub(r"[^a-z0-9']+", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

df_storico['chiave_titolo'] = df_storico['Title'].map(norm_titolo)
# alcuni titoli si ripetono nell'archivio: teniamo la prima occorrenza per
# evitare di duplicare righe nel merge
df_storico = df_storico.drop_duplicates(subset='chiave_titolo')[['chiave_titolo', 'Date']]
df_storico = df_storico.rename(columns={'Date': 'data_evento_storica'})

df_cfr['chiave_titolo'] = df_cfr['titolo'].map(norm_titolo)
df_cfr = df_cfr.merge(df_storico, on='chiave_titolo', how='left')

# priorita': data reale dall'archivio 2019 se disponibile, altrimenti la
# data di pubblicazione della voce attuale come stima approssimata (copre
# soprattutto gli incidenti 2020+, assenti dall'archivio del 2019)
df_cfr['fonte_data_evento'] = df_cfr['data_evento_storica'].notna().map(
    {True: 'archivio_cfr_2019', False: 'pubblicazione_stimata'}
)
df_cfr['data_evento'] = df_cfr['data_evento_storica'].fillna(df_cfr['data_pubblicazione'])
df_cfr = df_cfr.drop(columns=['chiave_titolo', 'data_evento_storica'])

print(df_cfr['fonte_data_evento'].value_counts())
print(df_cfr.shape)
df_cfr[['titolo', 'data_pubblicazione', 'data_evento', 'fonte_data_evento']].head(10)


In [14]:
# Test su un solo post, guardando il contenuto completo invece del solo excerpt
r = requests.get(f"{BASE}/posts/11049")  # Allanite
post = r.json()
print(pulisci_html(post["content"]["rendered"])[:2000])

This threat actor targets business and industrial control networks in the power-utility sector, for the purpose of espionage. In 2017, the U.S. Department of Homeland Security warned U.S. critical infrastructure operators about this threat actor and its capabilities.


In [16]:
nomi_cfr = {
    'Russian Federation': 'RUS', 'China': 'CHN',
    "Korea (Democratic People's Republic of)": 'PRK', 'Iran (Islamic Republic of)': 'IRN',
    'Ukraine': 'UKR', 'Sudan': 'SDN', 'South Sudan': 'SSD', 'Yemen': 'YEM',
    'Syrian Arab Republic': 'SYR', 'Ethiopia': 'ETH', 'Venezuela (Bolivarian Republic of)': 'VEN',
    'United States of America': 'USA', 'Israel': 'ISR', 'Korea (Republic of)': 'KOR',
    'Saudi Arabia': 'SAU', 'Italy': 'ITA', 'Estonia': 'EST'
}
# NOTA: controlla questi nomi contro df_cfr['sponsor_stato'] e df_cfr['paesi_vittima']
# uniti (split su ", "), stesso principio di controllo gia' fatto per UNHCR/World Bank
tutti_i_nomi_usati = set()
for col in ['sponsor_stato', 'paesi_vittima']:
    for cella in df_cfr[col].dropna():
        tutti_i_nomi_usati.update(x.strip() for x in cella.split(','))
print(sorted(tutti_i_nomi_usati))

['', 'Afghanistan', 'Albania', 'Algeria', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Bhutan', 'Bosnia and Herzegovina', 'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Cambodia', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Cuba', 'Cyprus', 'Czech Republic', 'Denmark', 'Djibouti', 'Ecuador', 'Egypt', 'El Salvador', 'Estonia', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Guam', 'Guatemala', 'Guyana', 'Hong Kong', 'Hungary', 'India', 'Indonesia', 'Iran (Islamic Republic of)', 'Iraq', 'Ireland', 'Israel', 'Italy', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya', 'Kiribati', "Korea (Democratic People's Republic of)", 'Korea (Republic of)', 'Kuwait', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Latvia', 'Lebanon', 'Libya', 'Lithuania', 'Luxembourg', 'Malaysia', 'Mali', 'Malta', 'Mexico', 'Mongolia', 'Montenegro', 'Morocco', 'Mozambique', 'Myanmar', 'Nepal', 'Netherlands'

In [18]:
import re
import html
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# PASSO 1 — Scarica i dati attuali dal sito nuovo di CFR (WordPress)
# ============================================================
BASE = "https://www.cfr.org/cyber-operations/wp-json/wp/v2"

def scarica_tassonomia(nome):
    """Scarica {id: nome} per una tassonomia (es. state_sponsor), paginando."""
    termini = {}
    pagina = 1
    while True:
        r = requests.get(f"{BASE}/{nome}", params={"per_page": 100, "page": pagina})
        if r.status_code != 200:
            break
        dati = r.json()
        if not dati:
            break
        for t in dati:
            termini[t["id"]] = t["name"]
        if pagina >= int(r.headers.get("X-WP-TotalPages", 1)):
            break
        pagina += 1
    return termini

tassonomie = {
    nome: scarica_tassonomia(nome)
    for nome in ["cyber_operation", "state_sponsor", "victim_category",
                 "victim_government_response", "victim"]
}

def pulisci_html(testo):
    # rimuove i tag e decodifica le entita' (es. &#8217; -> ') altrimenti i
    # titoli via API non combaciano con quelli puliti dell'archivio 2019 nel
    # merge del Passo 2, e restano leggibili nel CSV finale
    return html.unescape(re.sub('<[^<]+?>', '', testo or '')).strip()

incidenti = []
pagina = 1
while True:
    r = requests.get(f"{BASE}/posts", params={"per_page": 100, "page": pagina})
    if r.status_code != 200:
        break
    dati = r.json()
    if not dati:
        break
    for p in dati:
        incidenti.append({
            "id": p["id"],
            "data_pubblicazione": p["date"][:10],
            "titolo": pulisci_html(p["title"]["rendered"]),
            "categoria": ", ".join(tassonomie["cyber_operation"].get(i, "?") for i in p.get("cyber_operation", [])),
            "sponsor_stato": ", ".join(tassonomie["state_sponsor"].get(i, "?") for i in p.get("state_sponsor", [])),
            "categoria_vittima": ", ".join(tassonomie["victim_category"].get(i, "?") for i in p.get("victim_category", [])),
            "paesi_vittima": ", ".join(tassonomie["victim"].get(i, "?") for i in p.get("victim", [])),
            "risposta_governativa": ", ".join(tassonomie["victim_government_response"].get(i, "?") for i in p.get("victim_government_response", [])),
            "descrizione": pulisci_html(p["excerpt"]["rendered"]),
            "link": p["link"],
        })
    totpag = int(r.headers.get("X-WP-TotalPages", 1))
    if pagina >= totpag:
        break
    pagina += 1

df_cfr = pd.DataFrame(incidenti)
print(f"Passo 1 completato: {df_cfr.shape[0]} profili di minaccia scaricati")


# ============================================================
# PASSO 2 — Recupera le date vere dal vecchio sito, via Wayback Machine
# ============================================================
url_archivio = (
    "https://web.archive.org/web/20191022065745id_/"
    "https://www.cfr.org/interactive/cyber-operations/export-incidents"
)
df_storico = pd.read_csv(url_archivio)

def norm_titolo(s):
    s = str(s).strip().lower().replace('’', "'").replace('‘', "'")
    s = re.sub(r"[^a-z0-9']+", ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

df_storico['chiave_titolo'] = df_storico['Title'].map(norm_titolo)
df_storico = df_storico.drop_duplicates(subset='chiave_titolo')[['chiave_titolo', 'Date']]
df_storico = df_storico.rename(columns={'Date': 'data_evento'})

df_cfr['chiave_titolo'] = df_cfr['titolo'].map(norm_titolo)
righe_prima = df_cfr.shape[0]
df_cfr = df_cfr.merge(df_storico, on='chiave_titolo', how='left')
assert df_cfr.shape[0] == righe_prima, "Il merge ha duplicato righe, da controllare"

# Nessun fallback inventato: dove manca la data vera, resta vuota.
# Sara' l'LMM a provare a dedurla dal testo di 'descrizione', se possibile.
df_cfr['fonte_data_evento'] = df_cfr['data_evento'].notna().map(
    {True: 'archivio_cfr_2019', False: 'da_estrarre_con_llm'}
)
df_cfr = df_cfr.drop(columns=['chiave_titolo'])

print("Passo 2 completato:")
print(df_cfr['fonte_data_evento'].value_counts())


# ============================================================
# PASSO 3 — Tieni solo le righe che riguardano i nostri 17 paesi
# ============================================================
nomi_cfr = {
    'Russian Federation': 'RUS', 'China': 'CHN',
    "Korea (Democratic People's Republic of)": 'PRK', 'Iran (Islamic Republic of)': 'IRN',
    'Ukraine': 'UKR', 'Sudan': 'SDN', 'South Sudan': 'SSD', 'Yemen': 'YEM',
    'Syrian Arab Republic': 'SYR', 'Ethiopia': 'ETH', 'Venezuela (Bolivarian Republic of)': 'VEN',
    'United States of America': 'USA', 'United States': 'USA',
    'Israel': 'ISR', 'Korea (Republic of)': 'KOR',
    'Saudi Arabia': 'SAU', 'Italy': 'ITA', 'Estonia': 'EST'
}

def paesi_in_cella(cella, mappa):
    # match esatto sui nomi separati da virgola, non substring: "Sudan" e'
    # contenuto in "South Sudan" carattere per carattere, quindi un
    # controllo "in" scambierebbe le vittime Sud Sudan anche per Sudan
    if pd.isna(cella) or cella == '':
        return []
    nomi_nella_cella = {n.strip() for n in cella.split(',')}
    return sorted({iso3 for nome, iso3 in mappa.items() if nome in nomi_nella_cella})

df_cfr['sponsor_iso3'] = df_cfr['sponsor_stato'].apply(lambda x: paesi_in_cella(x, nomi_cfr))
df_cfr['vittime_iso3'] = df_cfr['paesi_vittima'].apply(lambda x: paesi_in_cella(x, nomi_cfr))

maschera_rilevante = (df_cfr['sponsor_iso3'].str.len() > 0) | (df_cfr['vittime_iso3'].str.len() > 0)
df_filtrato = df_cfr[maschera_rilevante].copy()

print(f"Passo 3 completato: {df_filtrato.shape[0]} profili rilevanti su {df_cfr.shape[0]} totali")


# ============================================================
# PASSO 4 — Salva un file per paese (come sponsor E come vittima)
# ============================================================
for iso3 in set(nomi_cfr.values()):
    e_sponsor = df_filtrato['sponsor_iso3'].apply(lambda lista: iso3 in lista)
    e_vittima = df_filtrato['vittime_iso3'].apply(lambda lista: iso3 in lista)
    sotto = df_filtrato[e_sponsor | e_vittima].copy()

    if sotto.empty:
        print(f"ATTENZIONE: nessun incidente CFR trovato per {iso3}")
        continue

    sotto['ruolo'] = sotto.apply(
        lambda r: 'entrambi' if (iso3 in r['sponsor_iso3'] and iso3 in r['vittime_iso3'])
        else ('attore' if iso3 in r['sponsor_iso3'] else 'vittima'),
        axis=1
    )

    out_dir = Path(f'../data/raw/cyber_advisories/{iso3}/')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{iso3}_cfr-cyber-operations.csv'
    colonne_da_salvare = ['titolo', 'categoria', 'sponsor_stato', 'paesi_vittima',
                           'ruolo', 'data_evento', 'fonte_data_evento', 'descrizione', 'link']
    sotto[colonne_da_salvare].to_csv(out_path, index=False)

    n_con_data = sotto['data_evento'].notna().sum()
    print(f'{iso3}: {len(sotto)} incidenti salvati ({n_con_data} con data certa, '
          f'{len(sotto)-n_con_data} da passare all\'LMM per la data)')


Passo 1 completato: 865 profili di minaccia scaricati
Passo 2 completato:
fonte_data_evento
da_estrarre_con_llm    603
archivio_cfr_2019      262
Name: count, dtype: int64
Passo 3 completato: 771 profili rilevanti su 865 totali
RUS: 243 incidenti salvati (89 con data certa, 154 da passare all'LMM per la data)
PRK: 113 incidenti salvati (25 con data certa, 88 da passare all'LMM per la data)
VEN: 1 incidenti salvati (1 con data certa, 0 da passare all'LMM per la data)
IRN: 128 incidenti salvati (36 con data certa, 92 da passare all'LMM per la data)
ETH: 2 incidenti salvati (1 con data certa, 1 da passare all'LMM per la data)
UKR: 89 incidenti salvati (17 con data certa, 72 da passare all'LMM per la data)
SSD: 1 incidenti salvati (0 con data certa, 1 da passare all'LMM per la data)
ITA: 7 incidenti salvati (4 con data certa, 3 da passare all'LMM per la data)
SYR: 12 incidenti salvati (7 con data certa, 5 da passare all'LMM per la data)
SAU: 37 incidenti salvati (20 con data certa, 17 da p

In [22]:
import requests
import feedparser

# feedparser manda uno User-Agent generico che il WAF di CISA blocca con un
# 403 (pagina di errore HTML al posto dell'XML, feed.entries vuoto). Scarico
# il feed a mano con uno User-Agent da browser e passo il contenuto a
# feedparser invece dell'URL.
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
r = requests.get("https://www.cisa.gov/cybersecurity-advisories/all.xml", headers=headers)
feed = feedparser.parse(r.content)

print(f"Voci nel feed: {len(feed.entries)}")
print(f"Più vecchia: {feed.entries[-1].published}")
print(f"Più recente: {feed.entries[0].published}")


Voci nel feed: 30
Più vecchia: Thu, 25 Jun 26 12:00:00 +0000
Più recente: Fri, 10 Jul 26 12:00:00 +0000


In [23]:
import re

# Prendi i nomi dei gruppi di minaccia rilevanti dai tuoi CSV per paese già salvati
nomi_gruppi = set(df_filtrato['titolo'].str.lower())
nomi_paesi_ricerca = ['russia', 'china', 'iran', 'north korea', "korea, democratic",
                       'ukraine', 'sudan', 'yemen', 'syria', 'ethiopia', 'venezuela',
                       'united states', 'israel', 'south korea', 'saudi arabia',
                       'italy', 'estonia']

risultati = []
for entry in feed.entries:
    testo = (entry.title + ' ' + entry.get('summary', '')).lower()
    gruppi_trovati = [g for g in nomi_gruppi if g in testo]
    paesi_trovati = [p for p in nomi_paesi_ricerca if p in testo]
    if gruppi_trovati or paesi_trovati:
        risultati.append({
            'titolo': entry.title,
            'data': entry.published,
            'link': entry.link,
            'gruppi_menzionati': gruppi_trovati,
            'paesi_menzionati': paesi_trovati,
        })

import pandas as pd
df_cisa_rilevanti = pd.DataFrame(risultati)
print(f"Advisory rilevanti trovati: {len(df_cisa_rilevanti)} su {len(feed.entries)} totali")
df_cisa_rilevanti.head(10)

Advisory rilevanti trovati: 8 su 30 totali


,titolo,data,link,gruppi_menzionati,paesi_menzionati
0,OpenPLC v3,"Thu, 09 Jul 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[],[united states]
1,Labcenter Proteus 9,"Tue, 07 Jul 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[kingdom],[]
2,"Digi International PortServer TS, Digi One SP IA","Tue, 07 Jul 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[],[united states]
3,ST Engineering iDirect iQ-Series Terminals,"Thu, 02 Jul 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[],[united states]
4,Gardyn IoT Hub,"Thu, 02 Jul 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[],[united states]
5,StoneFly Storage Concentrator,"Tue, 30 Jun 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-advisorie...,[],[united states]
6,Russian Intelligence Services Continue to Targ...,"Fri, 26 Jun 26 12:00:00 +0000",https://www.cisa.gov/resources-tools/resources...,[],[russia]
7,OHIF Viewers DICOM,"Thu, 25 Jun 26 12:00:00 +0000",https://www.cisa.gov/news-events/ics-medical-a...,[],[united states]


### Archivio storico CISA per attore di stato-nazione (PDF)

Il feed RSS sopra copre solo le ultime ~30 advisory. CISA pero' cataloga le
sue Cybersecurity Advisory congiunte anche per **attore di stato-nazione**
(facet `advisory_nation_state_actor` su `/news-events/cybersecurity-advisories`):
Russia, China, North Korea, Iran - esattamente i 4 paesi "attore_cyber" del
progetto. L'archivio copre 2017-2026 e ogni advisory ha un PDF ospitato
direttamente da CISA, quindi qui prendiamo i PDF veri e propri (coerente
con l'approccio multimodale) invece di un puro riassunto testuale.

In [24]:
import html
import re
import time
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

# CISA cataloga le proprie Cybersecurity Advisory congiunte per attore di
# stato-nazione: sono esattamente i 4 paesi "attore_cyber" del progetto.
# L'archivio copre 2017-2026 (molto piu' ampio delle ultime ~30 voci del
# feed RSS) e ogni advisory ha un PDF scaricabile ospitato da CISA stessa.
ATTORI_CISA = {"RUS": 1037, "CHN": 1038, "PRK": 1039, "IRN": 1040}

# stessa finestra temporale usata per ACLED/UNHCR/World Bank altrove nel notebook
ANNO_MIN, ANNO_MAX = 2018, 2024

def numero_pagine(actor_id):
    r = requests.get(
        "https://www.cisa.gov/news-events/cybersecurity-advisories",
        params={"f[0]": f"advisory_nation_state_actor:{actor_id}"},
        headers=HEADERS,
    )
    pagine = [int(p) for p in re.findall(r"page=(\d+)", r.text)]
    return (max(pagine) + 1) if pagine else 1

def estrai_pagina(actor_id, pagina):
    r = requests.get(
        "https://www.cisa.gov/news-events/cybersecurity-advisories",
        params={"f[0]": f"advisory_nation_state_actor:{actor_id}", "page": pagina},
        headers=HEADERS,
    )
    blocchi = re.findall(r'<article[^>]*class="[^"]*c-teaser[^"]*".*?</article>', r.text, re.S)
    righe = []
    for b in blocchi:
        data = re.search(r'<time datetime="([^"]*)"', b)
        link = re.search(r'<h3 class="c-teaser__title">\s*<a href="([^"]*)"', b)
        titolo = re.search(r"<span>([^<]*)</span>", b)
        meta = re.search(r'c-teaser__meta">([^<]*)', b)
        righe.append({
            "data": data.group(1)[:10] if data else None,
            "titolo": html.unescape(titolo.group(1).strip()) if titolo else None,
            "link": "https://www.cisa.gov" + link.group(1) if link else None,
            "meta": meta.group(1).strip() if meta else None,
        })
    return righe

def trova_pdf(link_pagina):
    r = requests.get(link_pagina, headers=HEADERS)
    candidati = re.findall(r'href="(/sites/default/files/[^"]*\.pdf)"', r.text)
    return ("https://www.cisa.gov" + candidati[0]) if candidati else None

riepilogo_globale = []

for iso3, actor_id in ATTORI_CISA.items():
    n_pag = numero_pagine(actor_id)
    advisory = []
    for p in range(n_pag):
        advisory.extend(estrai_pagina(actor_id, p))
        time.sleep(0.3)

    advisory = [r for r in advisory if r["data"] and ANNO_MIN <= int(r["data"][:4]) <= ANNO_MAX]

    out_dir = Path(f"../data/raw/cisa/{iso3}/")
    out_dir.mkdir(parents=True, exist_ok=True)

    for row in advisory:
        pdf_url = trova_pdf(row["link"]) if row["link"] else None
        row["pdf_url"] = pdf_url
        if pdf_url:
            nome_file = f"{iso3}_" + unquote(pdf_url.split("/")[-1])
            row["pdf_locale"] = nome_file
            r = requests.get(pdf_url, headers=HEADERS)
            (out_dir / nome_file).write_bytes(r.content)
        else:
            row["pdf_locale"] = None
        time.sleep(0.3)

    df = pd.DataFrame(advisory)
    df.to_csv(out_dir / f"{iso3}_cisa-advisories.csv", index=False)
    n_pdf = df["pdf_locale"].notna().sum()
    print(f"{iso3}: {len(df)} advisory ({n_pdf} con PDF scaricato, {len(df)-n_pdf} solo metadati)")
    riepilogo_globale.append((iso3, len(df), n_pdf))

print()
print("Totale:", sum(r[1] for r in riepilogo_globale), "advisory,", sum(r[2] for r in riepilogo_globale), "PDF scaricati")


RUS: 18 advisory (13 con PDF scaricato, 5 solo metadati)
CHN: 14 advisory (4 con PDF scaricato, 10 solo metadati)
PRK: 8 advisory (6 con PDF scaricato, 2 solo metadati)
IRN: 13 advisory (9 con PDF scaricato, 4 solo metadati)

Totale: 53 advisory, 32 PDF scaricati


### ENISA Threat Landscape + Microsoft Digital Defense Report

Due fonti che chiudono la dimensione cyber: entrambe pubblicano un report
annuale globale/EU sullo stato delle minacce cyber (non spezzato per
paese come CFR/CISA - qui prendiamo i PDF interi, e sara' la fase di
estrazione LLM a tirarne fuori le menzioni per paese).

Entrambe hanno un buco nel 2019: ENISA non ha pubblicato un'edizione
dedicata quell'anno (il periodo gennaio 2019 - aprile 2020 e' coperto
dal report etichettato "2020", uscito come 22 sotto-report invece di uno
unico - qui prendiamo il piu' vicino a un overview, la "List of top 15
threats"). Microsoft non ha pubblicato nulla nel 2019, nella transizione
di brand dal vecchio "Security Intelligence Report" (fino al 2018,
volume 24) al nuovo "Digital Defense Report" (dal 2020).

In [28]:
import requests
from pathlib import Path

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# ENISA Threat Landscape (ETL): report annuale globale/EU, non per paese -
# nessuno split ISO3 qui, verra' processato dall'LLM in seguito per estrarre
# menzioni per paese. Il 2019 non ha un'edizione dedicata: il periodo
# gen2019-apr2020 e' coperto dal report "2020" (pubblicato come 22
# sotto-report invece di uno unico - qui prendiamo il piu' vicino a un
# overview, "List of top 15 threats").
ENISA_URLS = {
    2018: "https://www.enisa.europa.eu/sites/default/files/publications/WP2018%20O.1.2.1%20-%20ENISA%20Threat%20Landscape%202018.pdf",
    2020: "https://www.enisa.europa.eu/sites/default/files/publications/ETL2020%20-%20ENISA%20List%20of%20top%2015%20Threats%20A4.pdf",
    2021: "https://www.enisa.europa.eu/sites/default/files/publications/ENISA%20Threat%20Landscape%202021.pdf",
    2022: "https://www.enisa.europa.eu/sites/default/files/publications/ENISA%20Threat%20Landscape%202022.pdf",
    2023: "https://www.enisa.europa.eu/sites/default/files/publications/ENISA%20Threat%20Landscape%202023.pdf",
    2024: "https://www.enisa.europa.eu/sites/default/files/2024-11/ENISA%20Threat%20Landscape%202024_0.pdf",
}

# Microsoft Digital Defense Report (MDDR): stesso discorso, report globale
# annuale. Il 2018 e' ancora sotto il vecchio nome "Security Intelligence
# Report" (volume 24). Il 2019 manca del tutto: Microsoft non ha pubblicato
# un'edizione quell'anno, nella transizione di brand da SIR a MDDR (avvenuta
# nel 2020).
MDDR_URLS = {
    2018: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-product-and-services/security/pdf/sir-report-v24.pdf",
    2020: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-product-and-services/security/pdf/microsoft-digital-defense-report-2020-september.pdf",
    2021: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-brand/documents/FY21-Microsoft-Digital-Defense-Report.pdf",
    2022: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-brand/documents/microsoft-digital-defense-report-2022.pdf",
    2023: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-brand/documents/MDDR_FINAL_2023_1004.pdf",
    2024: "https://cdn-dynmedia-1.microsoft.com/is/content/microsoftcorp/microsoft/final/en-us/microsoft-brand/documents/Microsoft%20Digital%20Defense%20Report%202024%20(1).pdf",
}

def scarica_report(nome_fonte, urls, out_dir, prefisso_file):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for anno, url in sorted(urls.items()):
        r = requests.get(url, headers=HEADERS)
        out_path = out_dir / f"{prefisso_file}_{anno}.pdf"
        out_path.write_bytes(r.content)
        print(f"{nome_fonte} {anno}: {r.status_code}, {len(r.content)//1024} KB -> {out_path.name}")
    anni_mancanti = sorted(set(range(2018, 2025)) - urls.keys())
    if anni_mancanti:
        print(f"{nome_fonte}: nessuna edizione trovata per {anni_mancanti}")

scarica_report("ENISA", ENISA_URLS, "../data/raw/enisa/", "ENISA_Threat_Landscape")
print()
scarica_report("MDDR", MDDR_URLS, "../data/raw/microsoft_mddr/", "Microsoft_Digital_Defense_Report")


ENISA 2018: 200, 5137 KB -> ENISA_Threat_Landscape_2018.pdf
ENISA 2020: 200, 1677 KB -> ENISA_Threat_Landscape_2020.pdf
ENISA 2021: 200, 5372 KB -> ENISA_Threat_Landscape_2021.pdf
ENISA 2022: 200, 5086 KB -> ENISA_Threat_Landscape_2022.pdf
ENISA 2023: 200, 8124 KB -> ENISA_Threat_Landscape_2023.pdf
ENISA 2024: 200, 7744 KB -> ENISA_Threat_Landscape_2024.pdf
ENISA: nessuna edizione trovata per [2019]

MDDR 2018: 200, 4377 KB -> Microsoft_Digital_Defense_Report_2018.pdf
MDDR 2020: 200, 9921 KB -> Microsoft_Digital_Defense_Report_2020.pdf
MDDR 2021: 200, 15073 KB -> Microsoft_Digital_Defense_Report_2021.pdf
MDDR 2022: 200, 19944 KB -> Microsoft_Digital_Defense_Report_2022.pdf
MDDR 2023: 200, 19675 KB -> Microsoft_Digital_Defense_Report_2023.pdf
MDDR 2024: 200, 19139 KB -> Microsoft_Digital_Defense_Report_2024.pdf
MDDR: nessuna edizione trovata per [2019]


## Dimensione 6 — Contesto generale (Wikipedia)

A differenza delle altre dimensioni, questo non è un segnale quantitativo:
è una sintesi narrativa annuale per paese (leadership, eventi principali,
situazioni in corso) presa dalle pagine Wikipedia "YYYY in \<Country\>".

Serve un campo esplicito in `config/extraction_schema.json`
(`contesto_generale`) perché altrimenti sarebbe dato scaricato senza uno
scopo dichiarato nel contratto JSON del Blocco A - puro rumore. Lo scopo
dichiarato: contesto per l'LMM in fase di estrazione, e knowledge di base
per gli agenti del Blocco C (simulazione OASIS-inspired).

Granularità annuale (non trimestrale, coerente con la fonte): lo stesso
testo vale per tutti e 4 i trimestri dell'anno. Dalle pagine teniamo solo
le sezioni utili (Incumbents/Events/Demographics), tagliando la coda a
basso valore (Deaths, See also, References, External links).

In [30]:
import re
import time
from pathlib import Path

import requests

HEADERS = {"User-Agent": "ThesisResearchBot/1.0 (progetto di tesi accademico; contatto: giacomomaldarella9@gmail.com)"}

# Nome esatto usato da Wikipedia nel titolo "YYYY in <nome>" per ciascun paese
NOMI_WIKIPEDIA = {
    'RUS': 'Russia', 'CHN': 'China', 'PRK': 'North Korea', 'IRN': 'Iran',
    'UKR': 'Ukraine', 'SDN': 'Sudan', 'SSD': 'South Sudan', 'YEM': 'Yemen',
    'SYR': 'Syria', 'ETH': 'Ethiopia', 'VEN': 'Venezuela',
    'USA': 'the United States', 'ISR': 'Israel', 'KOR': 'South Korea',
    'SAU': 'Saudi Arabia', 'ITA': 'Italy', 'EST': 'Estonia',
}

ANNI = range(2018, 2025)

# Sezioni di coda a basso valore per il contesto geopolitico (nascite/morti
# di personaggi non rilevanti, sport, cultura/musica/premi, link vari,
# bibliografia) - tagliamo il testo alla prima che compare, tenendo solo
# Incumbents/Events/Politics/Society/Demographics e simili. Verificato a
# mano su tutti i 119 file che nessuna di queste compaia mai prima di
# contenuto rilevante (compare sempre per ultima, dopo Events).
SEZIONI_DA_TAGLIARE = [
    'Births', 'Deaths', 'Sports', 'Sport', 'Music', 'Art and entertainment',
    'Arts and entertainment', 'Popular culture', 'Culture', 'Prizes',
    'Anniversaries', 'Holidays', 'See also', 'References', 'External links',
    'Further reading',
]

def pulisci_estratto(testo):
    # il "\n" di chiusura e' opzionale: se la sezione e' vuota ed e' l'ultima
    # della pagina (es. "References" senza testo), il markup finisce li'
    # senza newline successivo
    posizioni = [m.start() for sezione in SEZIONI_DA_TAGLIARE
                 for m in re.finditer(rf'\n== {re.escape(sezione)} ==(?:\n|$)', testo)]
    if posizioni:
        testo = testo[:min(posizioni)]
    return testo.strip()

def scarica_pagina(titolo, tentativi=3):
    for tentativo in range(tentativi):
        r = requests.get("https://en.wikipedia.org/w/api.php", params={
            "action": "query", "titles": titolo, "format": "json", "redirects": 1,
            "prop": "extracts", "explaintext": 1,
        }, headers=HEADERS, timeout=15)
        if r.status_code == 429:
            attesa = int(r.headers.get("retry-after", 10))
            time.sleep(attesa)
            continue
        r.raise_for_status()
        pagina = list(r.json()["query"]["pages"].values())[0]
        if "missing" in pagina:
            return None
        return pagina.get("extract", "")
    return None

riepilogo = []
for iso3, nome in NOMI_WIKIPEDIA.items():
    out_dir = Path(f"../data/raw/wikipedia/{iso3}/")
    out_dir.mkdir(parents=True, exist_ok=True)
    trovati = 0
    for anno in ANNI:
        titolo = f"{anno} in {nome}"
        estratto = scarica_pagina(titolo)
        if not estratto:
            print(f"ATTENZIONE: nessuna pagina Wikipedia trovata per '{titolo}'")
            continue
        testo_pulito = pulisci_estratto(estratto)
        out_path = out_dir / f"{iso3}_{anno}_wikipedia-context.txt"
        out_path.write_text(testo_pulito, encoding="utf-8")
        trovati += 1
        time.sleep(1.2)
    print(f"{iso3}: {trovati}/{len(list(ANNI))} anni salvati")
    riepilogo.append((iso3, trovati))

print()
print("Totale pagine salvate:", sum(t for _, t in riepilogo), "su", len(NOMI_WIKIPEDIA) * len(list(ANNI)))


RUS: 7/7 anni salvati
CHN: 7/7 anni salvati
PRK: 7/7 anni salvati
IRN: 7/7 anni salvati
UKR: 7/7 anni salvati
SDN: 7/7 anni salvati
SSD: 7/7 anni salvati
YEM: 7/7 anni salvati
SYR: 7/7 anni salvati
ETH: 7/7 anni salvati
VEN: 7/7 anni salvati
USA: 7/7 anni salvati
ISR: 7/7 anni salvati
KOR: 7/7 anni salvati
SAU: 7/7 anni salvati
ITA: 7/7 anni salvati
EST: 7/7 anni salvati

Totale pagine salvate: 119 su 119


## Controllo copertura finale

Prima di passare al notebook 02, verificare quali combinazioni (paese, fonte) risultano scoperte.

In [ ]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH)
copertura = manifest.groupby(['fonte', 'paese_iso3']).size().unstack(fill_value=0)
copertura